# 03 - Baseline for submission

## 1. Load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"
SUB_DIR = OUTPUTS / "submissions"
CV_DIR = OUTPUTS / "cv_results"

SUB_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)

daily_panel = pd.read_parquet(DATA_PROCESSED / "daily_panel.parquet")
sku_activity = pd.read_parquet(DATA_PROCESSED / "sku_activity.parquet")
sample = pd.read_csv(DATA_RAW / "sample_submission.csv")

daily_panel["Date"] = pd.to_datetime(daily_panel["Date"])

for col in [
    "first_sale_date", "last_sale_date",
    "first_transaction_date", "last_transaction_date"
]:
    if col in sku_activity.columns:
        sku_activity[col] = pd.to_datetime(sku_activity[col])

print("daily_panel:", daily_panel.shape)
print("sku_activity:", sku_activity.shape)
print("sample:", sample.shape)

display(daily_panel.head())
display(sku_activity.head())
display(sample.head())

daily_panel: (28014888, 17)
sku_activity: (15972, 20)
sample: (31944, 29)


,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip,dayofweek,is_saturday,is_sunday,month,day,weekofyear,year
0,SKU-00001,2020-11-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,11,17,47,2020
1,SKU-00001,2020-11-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0,0,11,18,47,2020
2,SKU-00001,2020-11-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,0,0,11,19,47,2020
3,SKU-00001,2020-11-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,0,11,20,47,2020
4,SKU-00001,2020-11-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,0,11,21,47,2020


,ItemCode,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
0,SKU-00001,15,15,30.0,30.0,0.0,3.608433e+07,0.0,3.608433e+07,30.0,2025-05-26,2025-08-28,2025-05-26,2025-08-28,1,8,8,3.608433e+07,781,0.0
1,SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.0,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.0
2,SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.0,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.0
3,SKU-00004,279,279,659.0,659.0,0.0,8.359968e+08,0.0,8.359968e+08,659.0,2023-07-19,2024-12-20,2023-07-19,2024-12-20,1,259,259,8.359968e+08,12,0.0
4,SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.0,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.0


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [2]:
print("Date range:", daily_panel["Date"].min(), "→", daily_panel["Date"].max())
print("Number of SKUs:", daily_panel["ItemCode"].nunique())
print("Sample rows:", len(sample))
print("Sample IDs unique:", sample["id"].nunique())

assert len(sample) == sample["id"].nunique()

Date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
Number of SKUs: 15972
Sample rows: 31944
Sample IDs unique: 31944


## 2. Config last forecast

In [3]:
FINAL_TRAIN_END = pd.Timestamp("2025-09-05")
FINAL_FORECAST_START = pd.Timestamp("2025-09-06")
HORIZON = 56

TARGET_FOR_PRED = "y_net_clip"  # prediction không được âm
TARGET_FOR_METRIC = "y_net"     # local WRMSSE đang dùng y_net

forecast_dates = pd.date_range(FINAL_FORECAST_START, periods=HORIZON, freq="D")

print(forecast_dates[0], "→", forecast_dates[-1])
print("Day of week start:", forecast_dates[0].day_name())

2025-09-06 00:00:00 → 2025-10-31 00:00:00
Day of week start: Saturday


## 3. Baseline component 1: recent mean

In [4]:
def make_recent_blend_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    windows=(28, 56, 112),
    weights=(0.50, 0.30, 0.20)
) -> pd.DataFrame:
    """
    Predict each future day by blended recent mean per SKU.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    itemcodes = sorted(panel["ItemCode"].unique())
    
    assert len(windows) == len(weights)
    assert abs(sum(weights) - 1.0) < 1e-9
    
    blended = pd.Series(0.0, index=itemcodes)
    
    for window, weight in zip(windows, weights):
        hist_start = train_end - pd.Timedelta(days=window - 1)
        
        hist = panel.loc[
            (panel["Date"] >= hist_start) &
            (panel["Date"] <= train_end),
            ["ItemCode", target_col]
        ]
        
        mean_by_sku = hist.groupby("ItemCode")[target_col].mean()
        blended += weight * mean_by_sku.reindex(itemcodes).fillna(0)
    
    pred = pd.DataFrame(
        np.repeat(blended.to_numpy()[:, None], horizon, axis=1),
        index=itemcodes,
        columns=forecast_dates
    )
    
    return pred

## 4. Baseline compononent 2: same day-of-week mean

In [5]:
def make_same_dow_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    window_days: int = 182
) -> pd.DataFrame:
    """
    For each future date, predict using SKU's historical mean on the same day-of-week
    within the recent window.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    hist_start = train_end - pd.Timedelta(days=window_days - 1)
    itemcodes = sorted(panel["ItemCode"].unique())
    
    hist = panel.loc[
        (panel["Date"] >= hist_start) &
        (panel["Date"] <= train_end),
        ["ItemCode", "dayofweek", target_col]
    ].copy()
    
    dow_mean = (
        hist.groupby(["ItemCode", "dayofweek"])[target_col]
        .mean()
        .unstack("dayofweek")
        .reindex(index=itemcodes, columns=range(7))
        .fillna(0)
    )
    
    pred = pd.DataFrame(index=itemcodes, columns=forecast_dates, dtype=float)
    
    for d in forecast_dates:
        pred[d] = dow_mean[d.dayofweek].to_numpy()
    
    return pred

## 5. Baseline component 3: yearly lag

In [6]:
def make_yearly_lag_prediction(
    panel: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    lags=(364, 728)
) -> pd.DataFrame:
    """
    Predict using values from approximately same weekday in previous years.
    lag=364 keeps weekday alignment.
    """
    train_end = pd.Timestamp(train_end)
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=horizon, freq="D")
    
    itemcodes = sorted(panel["ItemCode"].unique())
    
    pred_sum = pd.DataFrame(0.0, index=itemcodes, columns=forecast_dates)
    pred_count = pd.DataFrame(0.0, index=itemcodes, columns=forecast_dates)
    
    needed_hist_dates = []
    for lag in lags:
        needed_hist_dates.extend([d - pd.Timedelta(days=lag) for d in forecast_dates])
    
    needed_hist_dates = sorted(set(needed_hist_dates))
    
    hist = panel.loc[
        panel["Date"].isin(needed_hist_dates),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_wide = (
        hist.pivot(index="ItemCode", columns="Date", values=target_col)
        .reindex(index=itemcodes)
        .fillna(0)
    )
    
    for d in forecast_dates:
        for lag in lags:
            hist_date = d - pd.Timedelta(days=lag)
            
            if hist_date <= train_end and hist_date in hist_wide.columns:
                pred_sum[d] += hist_wide[hist_date].to_numpy()
                pred_count[d] += 1
    
    pred = pred_sum / pred_count.replace(0, np.nan)
    pred = pred.fillna(0)
    
    return pred

## 6. Cap prediction function

In [7]:
def compute_sku_caps(
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    target_col: str = "y_net_clip"
) -> pd.Series:
    """
    Compute SKU-level upper cap for predictions.
    Conservative cap for sparse SKU, looser cap for high-profit SKU.
    """
    train_end = pd.Timestamp(train_end)
    itemcodes = sorted(panel["ItemCode"].unique())
    
    hist_365 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=364)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_112 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=111)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    hist_56 = panel.loc[
        (panel["Date"] >= train_end - pd.Timedelta(days=55)) &
        (panel["Date"] <= train_end),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    # positive-day quantile, avoids full-calendar quantile becoming 0 for sparse SKU
    positive_hist = hist_365[hist_365[target_col] > 0]
    
    pos_q95 = positive_hist.groupby("ItemCode")[target_col].quantile(0.95)
    max_56 = hist_56.groupby("ItemCode")[target_col].max()
    mean_112 = hist_112.groupby("ItemCode")[target_col].mean()
    
    cap = pd.DataFrame(index=itemcodes)
    cap["pos_q95"] = pos_q95.reindex(itemcodes).fillna(0)
    cap["max_56"] = max_56.reindex(itemcodes).fillna(0)
    cap["mean_112"] = mean_112.reindex(itemcodes).fillna(0)
    
    cap["base_cap"] = np.maximum.reduce([
        cap["pos_q95"] * 2.0,
        cap["max_56"] * 1.2,
        cap["mean_112"] * 5.0
    ])
    
    meta = sku_activity.set_index("ItemCode").reindex(itemcodes)
    
    # Nếu SKU từng bán gần đây, cap tối thiểu là 1 để không bóp quá tay.
    recently_active = meta["days_since_last_sale"].fillna(9999) <= 56
    cap.loc[recently_active, "base_cap"] = cap.loc[recently_active, "base_cap"].clip(lower=1.0)
    
    # Top profit SKU cap rộng hơn, tránh bóp mất spike thật.
    profit_rank = meta["profit_rank"].fillna(999999)
    
    cap_multiplier = pd.Series(1.0, index=itemcodes)
    cap_multiplier.loc[profit_rank <= 100] = 3.0
    cap_multiplier.loc[(profit_rank > 100) & (profit_rank <= 500)] = 2.0
    cap_multiplier.loc[(profit_rank > 500) & (profit_rank <= 1000)] = 1.5
    
    final_cap = cap["base_cap"] * cap_multiplier
    
    return final_cap.fillna(0)

## 7. Post-processing function

In [8]:
def postprocess_prediction(
    pred: pd.DataFrame,
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    sunday_factor: float = 0.0,
    apply_cap: bool = True
) -> pd.DataFrame:
    """
    Apply business/metric-aware rules:
    - non-negative
    - Sunday reduction
    - inactive SKU zero/shrink
    - upper cap
    """
    pred = pred.copy()
    pred[pred < 0] = 0
    
    # 1. Sunday correction
    sunday_cols = [c for c in pred.columns if pd.Timestamp(c).dayofweek == 6]
    if sunday_cols:
        pred.loc[:, sunday_cols] *= sunday_factor
    
    # 2. Inactive rules
    meta = sku_activity.set_index("ItemCode").reindex(pred.index)
    
    days_since = meta["days_since_last_sale"].fillna(9999)
    active_days = meta["active_days"].fillna(0)
    profit_rank = meta["profit_rank"].fillna(999999)
    
    # Zero hard: SKU lâu không bán + không thuộc nhóm profit quan trọng
    zero_mask = (
        ((days_since > 365) & (profit_rank > 500)) |
        ((days_since > 180) & (active_days <= 3) & (profit_rank > 100)) |
        ((days_since > 90) & (active_days <= 1) & (profit_rank > 100))
    )
    
    pred.loc[zero_mask, :] = 0
    
    # Shrink: SKU hơi inactive, nhưng không zero cứng
    shrink_mask = (
        (days_since > 90) &
        (days_since <= 365) &
        (active_days <= 5) &
        (profit_rank > 500) &
        (~zero_mask)
    )
    
    pred.loc[shrink_mask, :] *= 0.25
    
    # 3. Cap
    if apply_cap:
        caps = compute_sku_caps(
            panel=panel,
            sku_activity=sku_activity,
            train_end=train_end,
            target_col=TARGET_FOR_PRED
        ).reindex(pred.index).fillna(0)
        
        pred = pred.clip(upper=caps, axis=0)
    
    pred = pred.fillna(0)
    pred[pred < 0] = 0
    
    return pred

## 8. Build baseline forecast

In [9]:
def make_baseline_v1_forecast(
    panel: pd.DataFrame,
    sku_activity: pd.DataFrame,
    train_end,
    forecast_start,
    horizon: int = 56,
    target_col: str = "y_net_clip",
    sunday_factor: float = 0.0
) -> pd.DataFrame:
    """
    Baseline v1:
    recent blend + same day-of-week + yearly lag + postprocess.
    """
    pred_recent = make_recent_blend_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        windows=(28, 56, 112),
        weights=(0.50, 0.30, 0.20)
    )
    
    pred_samedow = make_same_dow_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        window_days=182
    )
    
    pred_yearly = make_yearly_lag_prediction(
        panel,
        train_end=train_end,
        forecast_start=forecast_start,
        horizon=horizon,
        target_col=target_col,
        lags=(364, 728)
    )
    
    # Blend chính
    pred = (
        0.45 * pred_recent +
        0.40 * pred_samedow +
        0.15 * pred_yearly
    )
    
    pred = postprocess_prediction(
        pred,
        panel=panel,
        sku_activity=sku_activity,
        train_end=train_end,
        sunday_factor=sunday_factor,
        apply_cap=True
    )
    
    return pred

## 9. Local compute Validation

In [10]:
def compute_sku_metric_info(
    panel: pd.DataFrame,
    train_end,
    target_col: str = "y_net",
    profit_col: str = "profit",
    eps: float = 1e-8
) -> pd.DataFrame:
    """
    Compute SKU-level profit weight and RMSSE scale up to train_end.
    
    weight_i = max(sum(profit_i), 0) / total_positive_profit
    scale_i = mean((y_t - y_{t-1})^2) over training history
    """
    train_end = pd.Timestamp(train_end)
    
    hist = panel.loc[
        panel["Date"] <= train_end,
        ["ItemCode", "Date", target_col, profit_col]
    ].copy()
    
    hist = hist.sort_values(["ItemCode", "Date"])
    
    # 1. Profit weight
    profit_info = (
        hist.groupby("ItemCode", as_index=False)[profit_col]
        .sum()
        .rename(columns={profit_col: "total_profit_metric_train"})
    )
    
    profit_info["positive_profit"] = profit_info["total_profit_metric_train"].clip(lower=0)
    
    total_positive_profit = profit_info["positive_profit"].sum()
    
    if total_positive_profit <= 0:
        raise ValueError("Total positive profit is zero. Cannot compute weights.")
    
    profit_info["weight"] = profit_info["positive_profit"] / total_positive_profit
    
    # 2. RMSSE scale
    hist["diff"] = hist.groupby("ItemCode")[target_col].diff()
    hist["sq_diff"] = hist["diff"] ** 2
    
    scale_info = (
        hist.dropna(subset=["sq_diff"])
        .groupby("ItemCode", as_index=False)["sq_diff"]
        .mean()
        .rename(columns={"sq_diff": "scale"})
    )
    
    metric_info = profit_info.merge(scale_info, on="ItemCode", how="left")
    
    metric_info["scale"] = metric_info["scale"].fillna(0)
    metric_info["scale_safe"] = metric_info["scale"].clip(lower=eps)
    metric_info["zero_scale_flag"] = (metric_info["scale"] < eps).astype(int)
    
    metric_info["profit_rank"] = (
        metric_info["positive_profit"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    
    metric_info = metric_info.sort_values("profit_rank").reset_index(drop=True)
    metric_info["cum_weight"] = metric_info["weight"].cumsum()
    
    return metric_info

In [11]:
def make_actual_matrix(
    panel: pd.DataFrame,
    start_date,
    horizon: int = 56,
    target_col: str = "y_net"
) -> pd.DataFrame:
    """
    Create actual matrix:
    index = ItemCode
    columns = validation dates
    values = actual target
    """
    start_date = pd.Timestamp(start_date)
    dates = pd.date_range(start_date, periods=horizon, freq="D")
    
    subset = panel.loc[
        panel["Date"].isin(dates),
        ["ItemCode", "Date", target_col]
    ].copy()
    
    actual_wide = (
        subset.pivot(index="ItemCode", columns="Date", values=target_col)
        .sort_index()
        .reindex(columns=dates)
    )
    
    if actual_wide.isna().any().any():
        raise ValueError("Actual matrix contains NaN. Check date range or panel completeness.")
    
    return actual_wide

In [12]:
def wrmsse_score(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame,
    clip_pred: bool = True
):
    """
    Compute WRMSSE.
    
    actual_wide:
        index = ItemCode
        columns = forecast dates
    pred_wide:
        same shape as actual_wide
    metric_info:
        must contain ItemCode, weight, scale_safe
    """
    actual_wide = actual_wide.sort_index()
    
    pred_wide = (
        pred_wide
        .reindex(index=actual_wide.index, columns=actual_wide.columns)
        .fillna(0)
        .sort_index()
    )
    
    actual_values = actual_wide.to_numpy(dtype=float)
    pred_values = pred_wide.to_numpy(dtype=float)
    
    if clip_pred:
        pred_values = np.clip(pred_values, 0, None)
    
    mse = np.mean((actual_values - pred_values) ** 2, axis=1)
    
    detail = pd.DataFrame({
        "ItemCode": actual_wide.index,
        "mse": mse
    })
    
    detail = detail.merge(
        metric_info[[
            "ItemCode",
            "weight",
            "scale",
            "scale_safe",
            "total_profit_metric_train",
            "positive_profit",
            "profit_rank"
        ]],
        on="ItemCode",
        how="left"
    )
    
    if detail["weight"].isna().any():
        raise ValueError("Some SKUs in actual_wide are missing from metric_info.")
    
    detail["rmsse"] = np.sqrt(detail["mse"] / detail["scale_safe"])
    detail["weighted_rmsse"] = detail["weight"] * detail["rmsse"]
    
    score = detail["weighted_rmsse"].sum()
    
    return score, detail

In [13]:
def wrmsse_score_by_horizon_split(
    actual_wide: pd.DataFrame,
    pred_wide: pd.DataFrame,
    metric_info: pd.DataFrame
):
    actual_first = actual_wide.iloc[:, :28]
    pred_first = pred_wide.iloc[:, :28]
    
    actual_second = actual_wide.iloc[:, 28:56]
    pred_second = pred_wide.iloc[:, 28:56]
    
    score_first, detail_first = wrmsse_score(actual_first, pred_first, metric_info)
    score_second, detail_second = wrmsse_score(actual_second, pred_second, metric_info)
    
    return score_first, score_second, detail_first, detail_second

In [14]:
folds = [
    {
        "fold": "recent_2025",
        "train_end": "2025-07-11",
        "valid_start": "2025-07-12",
        "horizon": 56
    },
    {
        "fold": "seasonal_2024",
        "train_end": "2024-09-05",
        "valid_start": "2024-09-06",
        "horizon": 56
    },
    {
        "fold": "seasonal_2023",
        "train_end": "2023-09-05",
        "valid_start": "2023-09-06",
        "horizon": 56
    }
]

In [15]:
cv_records = []
detail_outputs = {}

for fold_cfg in folds:
    fold_name = fold_cfg["fold"]
    train_end = fold_cfg["train_end"]
    valid_start = fold_cfg["valid_start"]
    horizon = fold_cfg["horizon"]
    
    print("=" * 80)
    print("Fold:", fold_name)
    
    metric_info_fold = compute_sku_metric_info(
        daily_panel,
        train_end=train_end,
        target_col=TARGET_FOR_METRIC
    )
    
    actual_wide = make_actual_matrix(
        daily_panel,
        start_date=valid_start,
        horizon=horizon,
        target_col=TARGET_FOR_METRIC
    )
    
    pred_baseline_v1 = make_baseline_v1_forecast(
        daily_panel,
        sku_activity=sku_activity,
        train_end=train_end,
        forecast_start=valid_start,
        horizon=horizon,
        target_col=TARGET_FOR_PRED,
        sunday_factor=0.0
    )
    
    score_full, detail_full = wrmsse_score(
        actual_wide,
        pred_baseline_v1,
        metric_info_fold
    )
    
    score_first28, score_second28, _, _ = wrmsse_score_by_horizon_split(
        actual_wide,
        pred_baseline_v1,
        metric_info_fold
    )
    
    cv_records.append({
        "fold": fold_name,
        "model": "baseline_v1",
        "wrmsse_full56": score_full,
        "wrmsse_first28": score_first28,
        "wrmsse_second28": score_second28
    })
    
    detail_outputs[fold_name] = detail_full
    
    print("WRMSSE full 56:", score_full)
    print("WRMSSE first 28:", score_first28)
    print("WRMSSE second 28:", score_second28)

cv_baseline_v1 = pd.DataFrame(cv_records)
display(cv_baseline_v1)

cv_baseline_v1.to_csv(CV_DIR / "step4_baseline_v1_cv.csv", index=False)

Fold: recent_2025
WRMSSE full 56: 0.5892320943666405
WRMSSE first 28: 0.5304228644368046
WRMSSE second 28: 0.5643596962703616
Fold: seasonal_2024
WRMSSE full 56: 0.753451080472963
WRMSSE first 28: 0.6611441848265287
WRMSSE second 28: 0.7262940746721616
Fold: seasonal_2023
WRMSSE full 56: 0.8165782164719191
WRMSSE first 28: 0.6456654384060494
WRMSSE second 28: 0.8347668397028323


,fold,model,wrmsse_full56,wrmsse_first28,wrmsse_second28
0,recent_2025,baseline_v1,0.589232,0.530423,0.564360
1,seasonal_2024,baseline_v1,0.753451,0.661144,0.726294
2,seasonal_2023,baseline_v1,0.816578,0.645665,0.834767


## 10. Diagnostic errors by bucket

In [16]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "rank_001_100"
    elif rank <= 500:
        return "rank_101_500"
    elif rank <= 1000:
        return "rank_501_1000"
    elif rank <= 2000:
        return "rank_1001_2000"
    else:
        return "rank_2001_plus"

diagnostic_detail = detail_outputs["recent_2025"].copy()
diagnostic_detail["rank_bucket"] = diagnostic_detail["profit_rank"].apply(assign_rank_bucket)

bucket_summary = (
    diagnostic_detail.groupby("rank_bucket", as_index=False)
    .agg(
        sku_count=("ItemCode", "count"),
        weight_sum=("weight", "sum"),
        wrmsse_contribution=("weighted_rmsse", "sum"),
        avg_rmsse=("rmsse", "mean")
    )
    .sort_values("wrmsse_contribution", ascending=False)
)

display(bucket_summary)

bucket_summary.to_csv(CV_DIR / "step4_baseline_v1_recent_2025_bucket_summary.csv", index=False)

display(
    diagnostic_detail
    .sort_values("weighted_rmsse", ascending=False)
    .head(30)
)

,rank_bucket,sku_count,weight_sum,wrmsse_contribution,avg_rmsse
0,rank_001_100,100,0.397870,0.276372,0.496024
2,rank_101_500,400,0.235905,0.138883,0.591187
4,rank_501_1000,500,0.114096,0.062181,0.529765
3,rank_2001_plus,13972,0.146865,0.061456,89.399477
1,rank_1001_2000,1000,0.105264,0.050341,0.471500


,ItemCode,mse,weight,scale,scale_safe,total_profit_metric_train,positive_profit,profit_rank,rmsse,weighted_rmsse,rank_bucket
2,SKU-00003,89.743156,0.095070,56.281674,56.281674,1.587671e+10,1.587671e+10,1,1.262749,0.120049,rank_001_100
1,SKU-00002,28.775618,0.045248,22.822039,22.822039,7.556427e+09,7.556427e+09,2,1.122885,0.050808,rank_001_100
13993,SKU-14323,5314.945763,0.005414,1509.051267,1509.051267,9.041417e+08,9.041417e+08,10,1.876711,0.010161,rank_001_100
15241,SKU-15599,17830.028665,0.002707,3148.621685,3148.621685,4.520825e+08,4.520825e+08,26,2.379665,0.006442,rank_001_100
13990,SKU-14320,2167.502615,0.005120,1623.653506,1623.653506,8.549961e+08,8.549961e+08,11,1.155402,0.005915,rank_001_100
10248,SKU-10532,140.823261,0.002977,70.189747,70.189747,4.972008e+08,4.972008e+08,22,1.416447,0.004217,rank_001_100
11100,SKU-11398,1688.005216,0.000676,47.646435,47.646435,1.128099e+08,1.128099e+08,222,5.952120,0.004021,rank_101_500
8616,SKU-08863,1659.878745,0.003742,1627.361226,1627.361226,6.249812e+08,6.249812e+08,17,1.009941,0.003780,rank_001_100
6593,SKU-06772,183.064712,0.000679,6.265174,6.265174,1.133406e+08,1.133406e+08,220,5.405499,0.003669,rank_101_500
9839,SKU-10117,1791.207998,0.001124,298.143783,298.143783,1.876291e+08,1.876291e+08,114,2.451095,0.002754,rank_101_500


## 11. Build final forecast 56 days

In [17]:
final_pred_56 = make_baseline_v1_forecast(
    daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    sunday_factor=0.0
)

print(final_pred_56.shape)
display(final_pred_56.head())

print("min pred:", final_pred_56.min().min())
print("max pred:", final_pred_56.max().max())
print("mean pred:", final_pred_56.values.mean())
print("total forecast qty:", final_pred_56.values.sum())

(15972, 56)


,2025-09-06,2025-09-07,2025-09-08,2025-09-09,2025-09-10,2025-09-11,2025-09-12,2025-09-13,2025-09-14,2025-09-15,2025-09-16,2025-09-17,2025-09-18,2025-09-19,2025-09-20,2025-09-21,2025-09-22,2025-09-23,2025-09-24,2025-09-25,2025-09-26,2025-09-27,2025-09-28,2025-09-29,2025-09-30,2025-10-01,2025-10-02,2025-10-03,2025-10-04,2025-10-05,2025-10-06,2025-10-07,2025-10-08,2025-10-09,2025-10-10,2025-10-11,2025-10-12,2025-10-13,2025-10-14,2025-10-15,2025-10-16,2025-10-17,2025-10-18,2025-10-19,2025-10-20,2025-10-21,2025-10-22,2025-10-23,2025-10-24,2025-10-25,2025-10-26,2025-10-27,2025-10-28,2025-10-29,2025-10-30,2025-10-31
SKU-00001,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038
SKU-00002,5.404574,0.0,7.106497,6.475728,6.731497,6.854574,6.925728,5.404574,0.0,7.556497,7.450728,6.806497,6.854574,6.550728,5.629574,0.0,7.406497,7.225728,7.856497,7.379574,7.375728,5.629574,0.0,8.531497,7.150728,7.106497,7.079574,6.625728,5.629574,0.0,7.031497,7.600728,6.881497,6.704574,7.300728,5.554574,0.0,7.631497,7.000728,7.406497,7.904574,6.550728,5.704574,0.0,7.106497,7.150728,7.406497,6.929574,7.525728,5.854574,0.0,7.631497,7.600728,7.556497,7.454574,6.700728
SKU-00003,9.784251,0.0,12.884251,11.888098,11.709251,12.790021,10.841944,10.159251,0.0,12.509251,11.738098,12.759251,13.090021,12.191944,10.309251,0.0,13.184251,11.813098,11.784251,13.390021,12.491944,10.609251,0.0,12.809251,10.688098,12.084251,12.265021,10.241944,10.009251,0.0,10.934251,10.988098,11.784251,13.015021,10.916944,10.759251,0.0,11.534251,11.438098,12.159251,12.865021,10.391944,9.934251,0.0,11.909251,10.688098,11.934251,12.790021,10.691944,10.234251,0.0,12.434251,12.563098,12.459251,13.540021,10.391944
SKU-00004,0.000000,0.0,0.000000,0.075000,0.225000,0.075000,0.300000,0.300000,0.0,0.225000,0.075000,0.375000,0.075000,0.375000,0.150000,0.0,0.525000,0.150000,0.300000,0.525000,0.225000,0.525000,0.0,0.525000,0.225000,0.150000,0.225000,0.225000,0.000000,0.0,0.225000,0.000000,0.375000,0.225000,0.300000,0.225000,0.0,0.150000,0.150000,0.450000,0.600000,0.075000,0.075000,0.0,0.225000,0.150000,0.450000,0.075000,0.525000,0.000000,0.0,0.375000,0.300000,0.375000,0.300000,0.150000
SKU-00005,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


min pred: 0.0
max pred: 868.6418337912088
mean pred: 0.08508228543089634
total forecast qty: 76100.31872252747


## 12. Convert forcast 56 days to Kaggle submission

In [18]:
def make_kaggle_submission(
    pred_56: pd.DataFrame,
    sample: pd.DataFrame,
    forecast_start,
    output_path=None
) -> pd.DataFrame:
    """
    Convert 56-day forecast matrix into Kaggle submission format:
    - <SKU>_validation: day 1-28
    - <SKU>_evaluation: day 29-56
    """
    forecast_start = pd.Timestamp(forecast_start)
    forecast_dates = pd.date_range(forecast_start, periods=56, freq="D")
    
    validation_dates = forecast_dates[:28]
    evaluation_dates = forecast_dates[28:]
    
    f_cols = [f"F{i}" for i in range(1, 29)]
    
    sub = sample.copy()
    sub[f_cols] = sub[f_cols].astype(float)
    
    # Parse id
    parsed = sub["id"].str.rsplit("_", n=1, expand=True)
    sub["ItemCode_tmp"] = parsed[0]
    sub["window_tmp"] = parsed[1]
    
    # Safety
    missing_skus = set(sub["ItemCode_tmp"]) - set(pred_56.index)
    if missing_skus:
        raise ValueError(f"Missing SKUs in prediction: {len(missing_skus)}")
    
    for idx, row in sub.iterrows():
        sku = row["ItemCode_tmp"]
        window = row["window_tmp"]
        
        if window == "validation":
            values = pred_56.loc[sku, validation_dates].to_numpy(dtype=float)
        elif window == "evaluation":
            values = pred_56.loc[sku, evaluation_dates].to_numpy(dtype=float)
        else:
            raise ValueError(f"Unknown window suffix: {window}")
        
        sub.loc[idx, f_cols] = values
    
    sub = sub.drop(columns=["ItemCode_tmp", "window_tmp"])
    
    # Final checks
    assert sub.shape == sample.shape
    assert set(sub["id"]) == set(sample["id"])
    assert sub["id"].nunique() == len(sub)
    
    values = sub[f_cols].to_numpy(dtype=float)
    
    if np.isnan(values).any():
        raise ValueError("Submission contains NaN.")
    
    if np.isinf(values).any():
        raise ValueError("Submission contains inf.")
    
    if (values < 0).any():
        raise ValueError("Submission contains negative values.")
    
    if output_path is not None:
        sub.to_csv(output_path, index=False)
        print("Saved:", output_path)
    
    return sub

In [19]:
submission_path = SUB_DIR / "submission_baseline_v1.csv"

submission_baseline_v1 = make_kaggle_submission(
    final_pred_56,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_path
)

display(submission_baseline_v1.head())
display(submission_baseline_v1.tail())

print(submission_baseline_v1.shape)

Saved: ../outputs/submissions/submission_baseline_v1.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038,0.112500,0.0,0.143269,0.204808,0.327885,0.174038,0.174038
1,SKU-00002_validation,5.404574,0.0,7.106497,6.475728,6.731497,6.854574,6.925728,5.404574,0.0,7.556497,7.450728,6.806497,6.854574,6.550728,5.629574,0.0,7.406497,7.225728,7.856497,7.379574,7.375728,5.629574,0.0,8.531497,7.150728,7.106497,7.079574,6.625728
2,SKU-00003_validation,9.784251,0.0,12.884251,11.888098,11.709251,12.790021,10.841944,10.159251,0.0,12.509251,11.738098,12.759251,13.090021,12.191944,10.309251,0.0,13.184251,11.813098,11.784251,13.390021,12.491944,10.609251,0.0,12.809251,10.688098,12.084251,12.265021,10.241944
3,SKU-00004_validation,0.000000,0.0,0.000000,0.075000,0.225000,0.075000,0.300000,0.300000,0.0,0.225000,0.075000,0.375000,0.075000,0.375000,0.150000,0.0,0.525000,0.150000,0.300000,0.525000,0.225000,0.525000,0.0,0.525000,0.225000,0.150000,0.225000,0.225000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
31939,SKU-16329_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31940,SKU-16330_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31941,SKU-16331_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31942,SKU-16332_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31943,SKU-16333_evaluation,0.0,0.0,0.0,0.0,0.0,0.0,0.075,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(31944, 29)


## 13. QA after creating submission

In [20]:
f_cols = [f"F{i}" for i in range(1, 29)]

print("Shape:", submission_baseline_v1.shape)
print("Unique IDs:", submission_baseline_v1["id"].nunique())
print("Missing values:", submission_baseline_v1[f_cols].isna().sum().sum())
print("Negative values:", (submission_baseline_v1[f_cols] < 0).sum().sum())
print("Total forecast:", submission_baseline_v1[f_cols].sum().sum())
print("Max forecast:", submission_baseline_v1[f_cols].max().max())

assert submission_baseline_v1.shape == sample.shape
assert submission_baseline_v1["id"].tolist() == sample["id"].tolist()
assert submission_baseline_v1[f_cols].isna().sum().sum() == 0
assert (submission_baseline_v1[f_cols] < 0).sum().sum() == 0

Shape: (31944, 29)
Unique IDs: 31944
Missing values: 0
Negative values: 0
Total forecast: 76100.31872252749
Max forecast: 868.6418337912088


In [21]:
validation_sunday_f = ["F2", "F9", "F16", "F23"]
evaluation_sunday_f = ["F2", "F9", "F16", "F23"]

validation_rows = submission_baseline_v1["id"].str.endswith("_validation")
evaluation_rows = submission_baseline_v1["id"].str.endswith("_evaluation")

print("Validation Sunday forecast total:")
print(submission_baseline_v1.loc[validation_rows, validation_sunday_f].sum().sum())

print("Evaluation Sunday forecast total:")
print(submission_baseline_v1.loc[evaluation_rows, evaluation_sunday_f].sum().sum())

Validation Sunday forecast total:
0.0
Evaluation Sunday forecast total:
0.0


## 14. Submission A

In [22]:
# Submission A: recent mean 28 + Sunday zero

pred_recent28 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(28,),
    weights=(1.0,)
)

pred_recent28 = postprocess_prediction(
    pred_recent28,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

print("recent28 total:", pred_recent28.values.sum())
print("recent28 max:", pred_recent28.max().max())
print("recent28 mean:", pred_recent28.values.mean())

sunday_cols = [c for c in pred_recent28.columns if pd.Timestamp(c).dayofweek == 6]
print("Sunday total:", pred_recent28[sunday_cols].sum().sum())

submission_recent28_path = SUB_DIR / "submission_recent28_sunday0.csv"

submission_recent28 = make_kaggle_submission(
    pred_recent28,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_recent28_path
)

display(submission_recent28.head())

recent28 total: 62516.57142857143
recent28 max: 83.5
recent28 mean: 0.06989527591652739
Sunday total: 0.0
Saved: ../outputs/submissions/submission_recent28_sunday0.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.285714,0.0,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.0,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.0,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.0,0.285714,0.285714,0.285714,0.285714,0.285714
1,SKU-00002_validation,3.500000,0.0,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,0.0,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,0.0,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,0.0,3.500000,3.500000,3.500000,3.500000,3.500000
2,SKU-00003_validation,12.714286,0.0,12.714286,12.714286,12.714286,12.714286,12.714286,12.714286,0.0,12.714286,12.714286,12.714286,12.714286,12.714286,12.714286,0.0,12.714286,12.714286,12.714286,12.714286,12.714286,12.714286,0.0,12.714286,12.714286,12.714286,12.714286,12.714286
3,SKU-00004_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


In [23]:
f_cols = [f"F{i}" for i in range(1, 29)]

print("Shape:", submission_recent28.shape)
print("Unique IDs:", submission_recent28["id"].nunique())
print("Missing:", submission_recent28[f_cols].isna().sum().sum())
print("Negative:", (submission_recent28[f_cols] < 0).sum().sum())
print("Total:", submission_recent28[f_cols].sum().sum())
print("Max:", submission_recent28[f_cols].max().max())

assert submission_recent28.shape == sample.shape
assert submission_recent28["id"].tolist() == sample["id"].tolist()
assert submission_recent28[f_cols].isna().sum().sum() == 0
assert (submission_recent28[f_cols] < 0).sum().sum() == 0

Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 62516.571428571435
Max: 83.5


## 15. Submission B

In [24]:
# Submission B: recent blend + same DOW, no yearly lag

pred_recent = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(28, 56, 112),
    weights=(0.50, 0.30, 0.20)
)

pred_samedow = make_same_dow_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    window_days=182
)

pred_no_yearly = (
    0.55 * pred_recent +
    0.45 * pred_samedow
)

pred_no_yearly = postprocess_prediction(
    pred_no_yearly,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

print("no_yearly total:", pred_no_yearly.values.sum())
print("no_yearly max:", pred_no_yearly.max().max())
print("no_yearly mean:", pred_no_yearly.values.mean())

sunday_cols = [c for c in pred_no_yearly.columns if pd.Timestamp(c).dayofweek == 6]
print("Sunday total:", pred_no_yearly[sunday_cols].sum().sum())

submission_no_yearly_path = SUB_DIR / "submission_baseline_no_yearly.csv"

submission_no_yearly = make_kaggle_submission(
    pred_no_yearly,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_no_yearly_path
)

display(submission_no_yearly.head())

no_yearly total: 71715.32571428572
no_yearly max: 158.76352335164836
no_yearly mean: 0.08017974056639937
Sunday total: 0.0
Saved: ../outputs/submissions/submission_baseline_no_yearly.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731,0.137500,0.0,0.172115,0.241346,0.379808,0.206731,0.206731
1,SKU-00002_validation,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975,5.869052,0.0,6.855591,6.820975,6.855591,6.994052,6.820975
2,SKU-00003_validation,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359,11.272205,0.0,11.722205,11.445282,12.172205,12.881820,10.943359
3,SKU-00004_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


## 16. Build recent baseline variant function

In [25]:
def build_recent_baseline_submission(
    name: str,
    windows,
    weights,
    apply_cap: bool = True,
    sunday_factor: float = 0.0
):
    """
    Build recent-window baseline submission.
    Example:
    - windows=(21,), weights=(1.0,)
    - windows=(28, 56), weights=(0.75, 0.25)
    """
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=windows,
        weights=weights
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=sunday_factor,
        apply_cap=apply_cap
    )

    output_path = SUB_DIR / f"submission_{name}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        forecast_start=FINAL_FORECAST_START,
        output_path=output_path
    )

    f_cols = [f"F{i}" for i in range(1, 29)]
    validation_rows = sub["id"].str.endswith("_validation")
    evaluation_rows = sub["id"].str.endswith("_evaluation")
    sunday_f = ["F2", "F9", "F16", "F23"]

    summary = {
        "name": name,
        "windows": str(windows),
        "weights": str(weights),
        "apply_cap": apply_cap,
        "sunday_factor": sunday_factor,
        "total_submission": sub[f_cols].sum().sum(),
        "validation_total": sub.loc[validation_rows, f_cols].sum().sum(),
        "evaluation_total": sub.loc[evaluation_rows, f_cols].sum().sum(),
        "max_pred": sub[f_cols].max().max(),
        "missing": sub[f_cols].isna().sum().sum(),
        "negative": (sub[f_cols] < 0).sum().sum(),
        "validation_sunday_total": sub.loc[validation_rows, sunday_f].sum().sum(),
        "evaluation_sunday_total": sub.loc[evaluation_rows, sunday_f].sum().sum(),
        "path": str(output_path)
    }

    return pred, sub, summary

Create candidate

In [26]:
candidate_configs = [
    # recent best performer, rebuild with cap and Sunday zero
    {
        "name": "recent28_sunday0_rebuild",
        "windows": (28,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Follow trend 
    {
        "name": "recent21_sunday0",
        "windows": (21,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Follow trend very closely, slightly risky
    {
        "name": "recent14_sunday0",
        "windows": (14,),
        "weights": (1.0,),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # More stable, but may miss some recent spikes
    {
        "name": "recent28_56blend_sunday0",
        "windows": (28, 56),
        "weights": (0.75, 0.25),
        "apply_cap": True,
        "sunday_factor": 0.0
    },

    # Even more stable, but may miss more recent spikes
    {
        "name": "recent21_28blend_sunday0",
        "windows": (21, 28),
        "weights": (0.50, 0.50),
        "apply_cap": True,
        "sunday_factor": 0.0
    },
]

candidate_preds = {}
candidate_subs = {}
candidate_summaries = []

for cfg in candidate_configs:
    pred, sub, summary = build_recent_baseline_submission(**cfg)
    candidate_preds[cfg["name"]] = pred
    candidate_subs[cfg["name"]] = sub
    candidate_summaries.append(summary)

candidate_summary_df = pd.DataFrame(candidate_summaries)
display(candidate_summary_df)

Saved: ../outputs/submissions/submission_recent28_sunday0_rebuild.csv
Saved: ../outputs/submissions/submission_recent21_sunday0.csv
Saved: ../outputs/submissions/submission_recent14_sunday0.csv
Saved: ../outputs/submissions/submission_recent28_56blend_sunday0.csv
Saved: ../outputs/submissions/submission_recent21_28blend_sunday0.csv


,name,windows,weights,apply_cap,sunday_factor,total_submission,validation_total,evaluation_total,max_pred,missing,negative,validation_sunday_total,evaluation_sunday_total,path
0,recent28_sunday0_rebuild,"(28,)","(1.0,)",True,0.0,62516.571429,31258.285714,31258.285714,83.500000,0,0,0.0,0.0,../outputs/submissions/submission_recent28_sun...
1,recent21_sunday0,"(21,)","(1.0,)",True,0.0,61099.428571,30549.714286,30549.714286,93.714286,0,0,0.0,0.0,../outputs/submissions/submission_recent21_sun...
2,recent14_sunday0,"(14,)","(1.0,)",True,0.0,56064.000000,28032.000000,28032.000000,85.428571,0,0,0.0,0.0,../outputs/submissions/submission_recent14_sun...
3,recent28_56blend_sunday0,"(28, 56)","(0.75, 0.25)",True,0.0,62833.071429,31416.535714,31416.535714,82.750000,0,0,0.0,0.0,../outputs/submissions/submission_recent28_56b...
4,recent21_28blend_sunday0,"(21, 28)","(0.5, 0.5)",True,0.0,61808.000000,30904.000000,30904.000000,88.607143,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28b...


## New model 19/5

In [27]:
def build_recent_candidate(
    name,
    windows,
    weights,
    apply_cap=True,
    sunday_factor=0.0
):
    """
    Build recent-window candidate submission.

    Example:
    - windows=(21, 28), weights=(0.75, 0.25)
    - windows=(18, 21, 28), weights=(0.25, 0.50, 0.25)
    """
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=windows,
        weights=weights
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=sunday_factor,
        apply_cap=apply_cap
    )

    output_path = SUB_DIR / f"submission_{name}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        forecast_start=FINAL_FORECAST_START,
        output_path=output_path
    )

    f_cols = [f"F{i}" for i in range(1, 29)]
    validation_rows = sub["id"].str.endswith("_validation")
    evaluation_rows = sub["id"].str.endswith("_evaluation")
    sunday_f = ["F2", "F9", "F16", "F23"]

    summary = {
        "name": name,
        "windows": str(windows),
        "weights": str(weights),
        "total": sub[f_cols].sum().sum(),
        "validation_total": sub.loc[validation_rows, f_cols].sum().sum(),
        "evaluation_total": sub.loc[evaluation_rows, f_cols].sum().sum(),
        "max_pred": sub[f_cols].max().max(),
        "missing": sub[f_cols].isna().sum().sum(),
        "negative": (sub[f_cols] < 0).sum().sum(),
        "validation_sunday_total": sub.loc[validation_rows, sunday_f].sum().sum(),
        "evaluation_sunday_total": sub.loc[evaluation_rows, sunday_f].sum().sum(),
        "path": str(output_path)
    }

    return pred, sub, summary

In [28]:
candidate_configs = [
    {
        "name": "recent21_28_w75_25_sunday0",
        "windows": (21, 28),
        "weights": (0.75, 0.25),
    },
    {
        "name": "recent21_28_w25_75_sunday0",
        "windows": (21, 28),
        "weights": (0.25, 0.75),
    },
    {
        "name": "recent18_21_28blend_sunday0",
        "windows": (18, 21, 28),
        "weights": (0.25, 0.50, 0.25),
    },
    {
        "name": "recent21_28_56blend_sunday0",
        "windows": (21, 28, 56),
        "weights": (0.45, 0.40, 0.15),
    },
]

candidate_preds = {}
candidate_subs = {}
candidate_summaries = []

for cfg in candidate_configs:
    pred, sub, summary = build_recent_candidate(
        name=cfg["name"],
        windows=cfg["windows"],
        weights=cfg["weights"],
        apply_cap=True,
        sunday_factor=0.0
    )

    candidate_preds[cfg["name"]] = pred
    candidate_subs[cfg["name"]] = sub
    candidate_summaries.append(summary)

candidate_summary_df = pd.DataFrame(candidate_summaries)
display(candidate_summary_df)

Saved: ../outputs/submissions/submission_recent21_28_w75_25_sunday0.csv
Saved: ../outputs/submissions/submission_recent21_28_w25_75_sunday0.csv
Saved: ../outputs/submissions/submission_recent18_21_28blend_sunday0.csv
Saved: ../outputs/submissions/submission_recent21_28_56blend_sunday0.csv


,name,windows,weights,total,validation_total,evaluation_total,max_pred,missing,negative,validation_sunday_total,evaluation_sunday_total,path
0,recent21_28_w75_25_sunday0,"(21, 28)","(0.75, 0.25)",61453.714286,30726.857143,30726.857143,91.160714,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...
1,recent21_28_w25_75_sunday0,"(21, 28)","(0.25, 0.75)",62162.285714,31081.142857,31081.142857,86.053571,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...
2,recent18_21_28blend_sunday0,"(18, 21, 28)","(0.25, 0.5, 0.25)",61299.523810,30649.761905,30649.761905,89.926587,0,0,0.0,0.0,../outputs/submissions/submission_recent18_21_...
3,recent21_28_56blend_sunday0,"(21, 28, 56)","(0.45, 0.4, 0.15)",62068.757143,31034.378571,31034.378571,87.646429,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...


Tạo segmented recent window v1

In [29]:
def make_raw_recent_pred(window):
    """
    Recent mean prediction before postprocess.
    """
    return make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=(window,),
        weights=(1.0,)
    )


pred18_raw = make_raw_recent_pred(18)
pred21_raw = make_raw_recent_pred(21)
pred28_raw = make_raw_recent_pred(28)
pred56_raw = make_raw_recent_pred(56)

print("Raw recent predictions created.")
print("recent18 total raw:", pred18_raw.values.sum())
print("recent21 total raw:", pred21_raw.values.sum())
print("recent28 total raw:", pred28_raw.values.sum())
print("recent56 total raw:", pred56_raw.values.sum())

Raw recent predictions created.
recent18 total raw: 70563.11111111111
recent21 total raw: 71282.66666666666
recent28 total raw: 72936.0
recent56 total raw: 74413.0


In [30]:
def build_segmented_recent_window_v1():
    """
    Segment-specific recent window blend.
    
    Strategy:
    - Top/active SKUs: more responsive recent window.
    - Medium SKUs: balanced recent21/recent28.
    - Long-tail SKUs: safer recent28-heavy.
    """
    item_index = pred21_raw.index

    meta = sku_activity.set_index("ItemCode").reindex(item_index)

    profit_rank = meta["profit_rank"].fillna(999999)
    days_since = meta["days_since_last_sale"].fillna(9999)
    active_days = meta["active_days"].fillna(0)

    # Start from safe long-tail baseline
    pred = 0.25 * pred21_raw + 0.75 * pred28_raw

    # Segment A: very important and recently active SKUs
    mask_top_active = (
        (profit_rank <= 500) &
        (days_since <= 56) &
        (active_days >= 10)
    )

    pred.loc[mask_top_active, :] = (
        0.35 * pred18_raw.loc[mask_top_active, :] +
        0.50 * pred21_raw.loc[mask_top_active, :] +
        0.15 * pred28_raw.loc[mask_top_active, :]
    )

    # Segment B: important SKUs but not as active
    mask_mid = (
        (profit_rank > 500) &
        (profit_rank <= 2000) &
        (days_since <= 180)
    )

    pred.loc[mask_mid, :] = (
        0.50 * pred21_raw.loc[mask_mid, :] +
        0.50 * pred28_raw.loc[mask_mid, :]
    )

    # Segment C: top SKUs but slightly inactive - do not overreact too much
    mask_top_inactive = (
        (profit_rank <= 500) &
        (days_since > 56) &
        (days_since <= 180)
    )

    pred.loc[mask_top_inactive, :] = (
        0.30 * pred21_raw.loc[mask_top_inactive, :] +
        0.70 * pred28_raw.loc[mask_top_inactive, :]
    )

    # Segment D: long-tail inactive stays conservative
    mask_longtail_inactive = (
        (profit_rank > 2000) &
        (days_since > 90)
    )

    pred.loc[mask_longtail_inactive, :] = pred28_raw.loc[mask_longtail_inactive, :]

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=0.0,
        apply_cap=True
    )

    return pred


pred_segmented_v1 = build_segmented_recent_window_v1()

submission_segmented_v1_path = SUB_DIR / "submission_segmented_recent_window_v1.csv"

submission_segmented_v1 = make_kaggle_submission(
    pred_segmented_v1,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_segmented_v1_path
)

f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_segmented_v1["id"].str.endswith("_validation")
evaluation_rows = submission_segmented_v1["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

segmented_summary = {
    "name": "segmented_recent_window_v1",
    "total": submission_segmented_v1[f_cols].sum().sum(),
    "validation_total": submission_segmented_v1.loc[validation_rows, f_cols].sum().sum(),
    "evaluation_total": submission_segmented_v1.loc[evaluation_rows, f_cols].sum().sum(),
    "max_pred": submission_segmented_v1[f_cols].max().max(),
    "missing": submission_segmented_v1[f_cols].isna().sum().sum(),
    "negative": (submission_segmented_v1[f_cols] < 0).sum().sum(),
    "validation_sunday_total": submission_segmented_v1.loc[validation_rows, sunday_f].sum().sum(),
    "evaluation_sunday_total": submission_segmented_v1.loc[evaluation_rows, sunday_f].sum().sum(),
    "path": str(submission_segmented_v1_path)
}

display(pd.DataFrame([segmented_summary]))

Saved: ../outputs/submissions/submission_segmented_recent_window_v1.csv


,name,total,validation_total,evaluation_total,max_pred,missing,negative,validation_sunday_total,evaluation_sunday_total,path
0,segmented_recent_window_v1,61335.704762,30667.852381,30667.852381,90.454365,0,0,0.0,0.0,../outputs/submissions/submission_segmented_re...


So sánh tất cả candidate

In [31]:
all_summaries = candidate_summaries + [segmented_summary]
all_summary_df = pd.DataFrame(all_summaries)

display(
    all_summary_df[
        [
            "name",
            "total",
            "validation_total",
            "evaluation_total",
            "max_pred",
            "missing",
            "negative",
            "validation_sunday_total",
            "evaluation_sunday_total",
            "path"
        ]
    ].sort_values("total")
)

,name,total,validation_total,evaluation_total,max_pred,missing,negative,validation_sunday_total,evaluation_sunday_total,path
2,recent18_21_28blend_sunday0,61299.523810,30649.761905,30649.761905,89.926587,0,0,0.0,0.0,../outputs/submissions/submission_recent18_21_...
4,segmented_recent_window_v1,61335.704762,30667.852381,30667.852381,90.454365,0,0,0.0,0.0,../outputs/submissions/submission_segmented_re...
0,recent21_28_w75_25_sunday0,61453.714286,30726.857143,30726.857143,91.160714,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...
3,recent21_28_56blend_sunday0,62068.757143,31034.378571,31034.378571,87.646429,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...
1,recent21_28_w25_75_sunday0,62162.285714,31081.142857,31081.142857,86.053571,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...


In [32]:
# Candidate: 0.30 * recent18 + 0.50 * recent21 + 0.20 * recent28

candidate_name = "recent18_21_28_w30_50_20_sunday0"

pred_w30_50_20 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(18, 21, 28),
    weights=(0.30, 0.50, 0.20)
)

pred_w30_50_20 = postprocess_prediction(
    pred_w30_50_20,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_w30_50_20_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_w30_50_20 = make_kaggle_submission(
    pred_w30_50_20,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_w30_50_20_path
)

display(submission_w30_50_20.head())
print("Saved:", submission_w30_50_20_path)

Saved: ../outputs/submissions/submission_recent18_21_28_w30_50_20_sunday0.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.300000,0.0,0.300000,0.300000,0.300000,0.300000,0.300000,0.300000,0.0,0.300000,0.300000,0.300000,0.300000,0.300000,0.300000,0.0,0.300000,0.300000,0.300000,0.300000,0.300000,0.300000,0.0,0.300000,0.300000,0.300000,0.300000,0.300000
1,SKU-00002_validation,3.028571,0.0,3.028571,3.028571,3.028571,3.028571,3.028571,3.028571,0.0,3.028571,3.028571,3.028571,3.028571,3.028571,3.028571,0.0,3.028571,3.028571,3.028571,3.028571,3.028571,3.028571,0.0,3.028571,3.028571,3.028571,3.028571,3.028571
2,SKU-00003_validation,13.154762,0.0,13.154762,13.154762,13.154762,13.154762,13.154762,13.154762,0.0,13.154762,13.154762,13.154762,13.154762,13.154762,13.154762,0.0,13.154762,13.154762,13.154762,13.154762,13.154762,13.154762,0.0,13.154762,13.154762,13.154762,13.154762,13.154762
3,SKU-00004_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
4,SKU-00005_validation,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000


Saved: ../outputs/submissions/submission_recent18_21_28_w30_50_20_sunday0.csv


In [33]:
f_cols = [f"F{i}" for i in range(1, 29)]

validation_rows = submission_w30_50_20["id"].str.endswith("_validation")
evaluation_rows = submission_w30_50_20["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Shape:", submission_w30_50_20.shape)
print("Unique IDs:", submission_w30_50_20["id"].nunique())
print("Missing:", submission_w30_50_20[f_cols].isna().sum().sum())
print("Negative:", (submission_w30_50_20[f_cols] < 0).sum().sum())
print("Total:", submission_w30_50_20[f_cols].sum().sum())
print("Max:", submission_w30_50_20[f_cols].max().max())
print("Validation Sunday total:", submission_w30_50_20.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_w30_50_20.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_w30_50_20.shape == sample.shape
assert submission_w30_50_20["id"].tolist() == sample["id"].tolist()
assert submission_w30_50_20[f_cols].isna().sum().sum() == 0
assert (submission_w30_50_20[f_cols] < 0).sum().sum() == 0

Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 61197.828571428574
Max: 90.19047619047619
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [34]:
extra_candidate_configs = [
    {
        "name": "recent18_21_28_w35_45_20_sunday0",
        "windows": (18, 21, 28),
        "weights": (0.35, 0.45, 0.20)
    },
    {
        "name": "recent18_21_28_w33_34_33_sunday0",
        "windows": (18, 21, 28),
        "weights": (0.33, 0.34, 0.33)
    },
    {
        "name": "recent21_28_56blend_sunday0",
        "windows": (21, 28, 56),
        "weights": (0.45, 0.40, 0.15)
    }
]

extra_summaries = []

for cfg in extra_candidate_configs:
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=cfg["windows"],
        weights=cfg["weights"]
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=0.0,
        apply_cap=True
    )

    output_path = SUB_DIR / f"submission_{cfg['name']}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        forecast_start=FINAL_FORECAST_START,
        output_path=output_path
    )

    validation_rows = sub["id"].str.endswith("_validation")
    evaluation_rows = sub["id"].str.endswith("_evaluation")

    summary = {
        "name": cfg["name"],
        "windows": cfg["windows"],
        "weights": cfg["weights"],
        "total": sub[f_cols].sum().sum(),
        "max": sub[f_cols].max().max(),
        "missing": sub[f_cols].isna().sum().sum(),
        "negative": (sub[f_cols] < 0).sum().sum(),
        "validation_sunday_total": sub.loc[validation_rows, sunday_f].sum().sum(),
        "evaluation_sunday_total": sub.loc[evaluation_rows, sunday_f].sum().sum(),
        "path": str(output_path)
    }

    extra_summaries.append(summary)

extra_summary_df = pd.DataFrame(extra_summaries)
display(extra_summary_df)

Saved: ../outputs/submissions/submission_recent18_21_28_w35_45_20_sunday0.csv
Saved: ../outputs/submissions/submission_recent18_21_28_w33_34_33_sunday0.csv
Saved: ../outputs/submissions/submission_recent21_28_56blend_sunday0.csv


,name,windows,weights,total,max,missing,negative,validation_sunday_total,evaluation_sunday_total,path
0,recent18_21_28_w35_45_20_sunday0,"(18, 21, 28)","(0.35, 0.45, 0.2)",61166.990476,89.943651,0,0,0.0,0.0,../outputs/submissions/submission_recent18_21_...
1,recent18_21_28_w33_34_33_sunday0,"(18, 21, 28)","(0.33, 0.34, 0.33)",61363.554286,88.714524,0,0,0.0,0.0,../outputs/submissions/submission_recent18_21_...
2,recent21_28_56blend_sunday0,"(21, 28, 56)","(0.45, 0.4, 0.15)",62068.757143,87.646429,0,0,0.0,0.0,../outputs/submissions/submission_recent21_28_...


In [36]:
candidate_name = "recent18_21_28_w35_50_15_sunday0"

pred_w35_50_15 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(18, 21, 28),
    weights=(0.35, 0.50, 0.15)
)

pred_w35_50_15 = postprocess_prediction(
    pred_w35_50_15,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_w35_50_15_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_w35_50_15 = make_kaggle_submission(
    pred_w35_50_15,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_w35_50_15_path
)

f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_w35_50_15["id"].str.endswith("_validation")
evaluation_rows = submission_w35_50_15["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_w35_50_15_path)
print("Shape:", submission_w35_50_15.shape)
print("Unique IDs:", submission_w35_50_15["id"].nunique())
print("Missing:", submission_w35_50_15[f_cols].isna().sum().sum())
print("Negative:", (submission_w35_50_15[f_cols] < 0).sum().sum())
print("Total:", submission_w35_50_15[f_cols].sum().sum())
print("Max:", submission_w35_50_15[f_cols].max().max())
print("Validation Sunday total:", submission_w35_50_15.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_w35_50_15.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_w35_50_15.shape == sample.shape
assert submission_w35_50_15["id"].tolist() == sample["id"].tolist()
assert submission_w35_50_15[f_cols].isna().sum().sum() == 0
assert (submission_w35_50_15[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_recent18_21_28_w35_50_15_sunday0.csv
Saved: ../outputs/submissions/submission_recent18_21_28_w35_50_15_sunday0.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 61096.133333333346
Max: 90.45436507936508
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [37]:
candidate_name = "recent18_21_28_w40_50_10_sunday0"

pred_w40_50_10 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(18, 21, 28),
    weights=(0.40, 0.50, 0.10)
)

pred_w40_50_10 = postprocess_prediction(
    pred_w40_50_10,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_w40_50_10_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_w40_50_10 = make_kaggle_submission(
    pred_w40_50_10,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_w40_50_10_path
)

f_cols = [f"F{i}" for i in range(1, 29)]

print("Saved:", submission_w40_50_10_path)
print("Shape:", submission_w40_50_10.shape)
print("Unique IDs:", submission_w40_50_10["id"].nunique())
print("Missing:", submission_w40_50_10[f_cols].isna().sum().sum())
print("Negative:", (submission_w40_50_10[f_cols] < 0).sum().sum())
print("Total:", submission_w40_50_10[f_cols].sum().sum())
print("Max:", submission_w40_50_10[f_cols].max().max())

assert submission_w40_50_10.shape == sample.shape
assert submission_w40_50_10["id"].tolist() == sample["id"].tolist()
assert submission_w40_50_10[f_cols].isna().sum().sum() == 0
assert (submission_w40_50_10[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_recent18_21_28_w40_50_10_sunday0.csv
Saved: ../outputs/submissions/submission_recent18_21_28_w40_50_10_sunday0.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 60994.4380952381
Max: 90.71825396825396


In [38]:
def create_recent_submission(name, windows, weights):
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=windows,
        weights=weights
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=FINAL_TRAIN_END,
        sunday_factor=0.0,
        apply_cap=True
    )

    output_path = SUB_DIR / f"submission_{name}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        forecast_start=FINAL_FORECAST_START,
        output_path=output_path
    )

    f_cols = [f"F{i}" for i in range(1, 29)]
    validation_rows = sub["id"].str.endswith("_validation")
    evaluation_rows = sub["id"].str.endswith("_evaluation")
    sunday_f = ["F2", "F9", "F16", "F23"]

    summary = {
        "name": name,
        "total": sub[f_cols].sum().sum(),
        "max": sub[f_cols].max().max(),
        "missing": sub[f_cols].isna().sum().sum(),
        "negative": (sub[f_cols] < 0).sum().sum(),
        "validation_sunday_total": sub.loc[validation_rows, sunday_f].sum().sum(),
        "evaluation_sunday_total": sub.loc[evaluation_rows, sunday_f].sum().sum(),
        "path": str(output_path)
    }

    print(summary)

    assert sub.shape == sample.shape
    assert sub["id"].tolist() == sample["id"].tolist()
    assert summary["missing"] == 0
    assert summary["negative"] == 0

    return pred, sub, summary


In [39]:
candidate_specs = [
    ("recent18_21_28_w45_50_05_sunday0", (18, 21, 28), (0.45, 0.50, 0.05)),
    ("recent18_21_w50_50_sunday0", (18, 21), (0.50, 0.50)),
    ("recent18_21_28_w40_50_10_sunday0", (18, 21, 28), (0.40, 0.50, 0.10)),
    ("recent18_21_28_56_private_safe", (18, 21, 28, 56), (0.30, 0.45, 0.20, 0.05)),
]

candidate_outputs = {}

for name, windows, weights in candidate_specs:
    pred, sub, summary = create_recent_submission(name, windows, weights)
    candidate_outputs[name] = {
        "pred": pred,
        "sub": sub,
        "summary": summary
    }

Saved: ../outputs/submissions/submission_recent18_21_28_w45_50_05_sunday0.csv
{'name': 'recent18_21_28_w45_50_05_sunday0', 'total': np.float64(60892.74285714286), 'max': np.float64(90.98214285714285), 'missing': np.int64(0), 'negative': np.int64(0), 'validation_sunday_total': np.float64(0.0), 'evaluation_sunday_total': np.float64(0.0), 'path': '../outputs/submissions/submission_recent18_21_28_w45_50_05_sunday0.csv'}
Saved: ../outputs/submissions/submission_recent18_21_w50_50_sunday0.csv
{'name': 'recent18_21_w50_50_sunday0', 'total': np.float64(60791.04761904762), 'max': np.float64(91.24603174603175), 'missing': np.int64(0), 'negative': np.int64(0), 'validation_sunday_total': np.float64(0.0), 'evaluation_sunday_total': np.float64(0.0), 'path': '../outputs/submissions/submission_recent18_21_w50_50_sunday0.csv'}
Saved: ../outputs/submissions/submission_recent18_21_28_w40_50_10_sunday0.csv
{'name': 'recent18_21_28_w40_50_10_sunday0', 'total': np.float64(60994.4380952381), 'max': np.float6

In [40]:
def make_raw_recent_pred(window):
    return make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=(window,),
        weights=(1.0,)
    )

pred14_raw = make_raw_recent_pred(14)
pred18_raw = make_raw_recent_pred(18)
pred21_raw = make_raw_recent_pred(21)
pred28_raw = make_raw_recent_pred(28)
pred56_raw = make_raw_recent_pred(56)

print("Raw predictions created.")
print("recent14 total:", pred14_raw.values.sum())
print("recent18 total:", pred18_raw.values.sum())
print("recent21 total:", pred21_raw.values.sum())
print("recent28 total:", pred28_raw.values.sum())
print("recent56 total:", pred56_raw.values.sum())

Raw predictions created.
recent14 total: 65408.0
recent18 total: 70563.11111111111
recent21 total: 71282.66666666666
recent28 total: 72936.0
recent56 total: 74413.0


In [41]:
candidate_name = "segmented_recent_top_active_v1"

item_index = pred21_raw.index
meta = sku_activity.set_index("ItemCode").reindex(item_index)

profit_rank = meta["profit_rank"].fillna(999999)
days_since = meta["days_since_last_sale"].fillna(9999)
active_days = meta["active_days"].fillna(0)

# Base = current global best-ish: recent18/recent21
pred_segmented = (
    0.50 * pred18_raw +
    0.50 * pred21_raw
)

# Group A: top-profit + very active/recent
# More responsive: include recent14, reduce older window
mask_top_active = (
    (profit_rank <= 500) &
    (days_since <= 45) &
    (active_days >= 10)
)

pred_segmented.loc[mask_top_active, :] = (
    0.25 * pred14_raw.loc[mask_top_active, :] +
    0.45 * pred18_raw.loc[mask_top_active, :] +
    0.30 * pred21_raw.loc[mask_top_active, :]
)

# Group B: top-profit but less active
# Keep some recent28 to avoid overreacting
mask_top_less_active = (
    (profit_rank <= 500) &
    (~mask_top_active)
)

pred_segmented.loc[mask_top_less_active, :] = (
    0.40 * pred18_raw.loc[mask_top_less_active, :] +
    0.45 * pred21_raw.loc[mask_top_less_active, :] +
    0.15 * pred28_raw.loc[mask_top_less_active, :]
)

# Group C: medium-weight SKUs
# Use proven stable global blend
mask_medium = (
    (profit_rank > 500) &
    (profit_rank <= 2000)
)

pred_segmented.loc[mask_medium, :] = (
    0.35 * pred18_raw.loc[mask_medium, :] +
    0.50 * pred21_raw.loc[mask_medium, :] +
    0.15 * pred28_raw.loc[mask_medium, :]
)

# Group D: long-tail active SKUs
# Slightly more conservative
mask_longtail_active = (
    (profit_rank > 2000) &
    (days_since <= 90)
)

pred_segmented.loc[mask_longtail_active, :] = (
    0.30 * pred21_raw.loc[mask_longtail_active, :] +
    0.50 * pred28_raw.loc[mask_longtail_active, :] +
    0.20 * pred56_raw.loc[mask_longtail_active, :]
)

# Group E: long-tail inactive
# Very conservative
mask_longtail_inactive = (
    (profit_rank > 2000) &
    (days_since > 90)
)

pred_segmented.loc[mask_longtail_inactive, :] = (
    0.20 * pred21_raw.loc[mask_longtail_inactive, :] +
    0.80 * pred28_raw.loc[mask_longtail_inactive, :]
)

# Apply existing postprocess: Sunday zero, inactive rules, caps
pred_segmented = postprocess_prediction(
    pred_segmented,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_segmented_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_segmented = make_kaggle_submission(
    pred_segmented,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_segmented_path
)

f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_segmented["id"].str.endswith("_validation")
evaluation_rows = submission_segmented["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_segmented_path)
print("Shape:", submission_segmented.shape)
print("Unique IDs:", submission_segmented["id"].nunique())
print("Missing:", submission_segmented[f_cols].isna().sum().sum())
print("Negative:", (submission_segmented[f_cols] < 0).sum().sum())
print("Total:", submission_segmented[f_cols].sum().sum())
print("Max:", submission_segmented[f_cols].max().max())
print("Validation Sunday total:", submission_segmented.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_segmented.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_segmented.shape == sample.shape
assert submission_segmented["id"].tolist() == sample["id"].tolist()
assert submission_segmented[f_cols].isna().sum().sum() == 0
assert (submission_segmented[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_segmented_recent_top_active_v1.csv
Saved: ../outputs/submissions/submission_segmented_recent_top_active_v1.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 60470.5619047619
Max: 89.42142857142856
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [42]:
# Candidate: recent18/21 base + mild weekday adjustment with shrinkage

candidate_name = "recent18_21_w50_50_weekday_shrink_v1"

# 1. Base prediction: current best global blend
pred_base = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(18, 21),
    weights=(0.50, 0.50)
)

# 2. Compute recent weekday factors from last 84 days
hist_start = FINAL_TRAIN_END - pd.Timedelta(days=83)

hist = daily_panel.loc[
    (daily_panel["Date"] >= hist_start) &
    (daily_panel["Date"] <= FINAL_TRAIN_END),
    ["ItemCode", "Date", TARGET_FOR_PRED, "dayofweek"]
].copy()

# Exclude Sunday from factor learning because Sunday is handled separately
hist_non_sun = hist[hist["dayofweek"] != 6].copy()

sku_mean = (
    hist_non_sun
    .groupby("ItemCode")[TARGET_FOR_PRED]
    .mean()
    .rename("sku_recent_mean")
)

dow_mean = (
    hist_non_sun
    .groupby(["ItemCode", "dayofweek"])[TARGET_FOR_PRED]
    .mean()
    .unstack("dayofweek")
    .reindex(columns=[0, 1, 2, 3, 4, 5])
)

# Raw factor = SKU's weekday mean / SKU's recent non-Sunday mean
factor = dow_mean.div(sku_mean, axis=0).replace([np.inf, -np.inf], np.nan).fillna(1.0)

# 3. Shrink factor toward 1 based on recent activity
active_recent = (
    hist_non_sun
    .groupby("ItemCode")[TARGET_FOR_PRED]
    .apply(lambda x: int((x > 0).sum()))
    .rename("active_recent_days")
)

meta = sku_activity.set_index("ItemCode").reindex(pred_base.index)
profit_rank = meta["profit_rank"].fillna(999999)

active_recent = active_recent.reindex(pred_base.index).fillna(0)

# Confidence: only active SKUs get stronger weekday adjustment
confidence = (active_recent / 12).clip(0, 1)

# Top-profit SKUs can use slightly more weekday signal
confidence = confidence.where(profit_rank > 1000, confidence.clip(lower=0.35))

factor = factor.reindex(pred_base.index).fillna(1.0)

# Shrink toward 1
factor_shrunk = 1.0 + factor.sub(1.0).mul(confidence, axis=0)

# Clip to avoid noisy SKU-level weekday spikes
# top SKUs can move a bit more; tail SKUs move less
factor_top = factor_shrunk.clip(lower=0.70, upper=1.30)
factor_tail = factor_shrunk.clip(lower=0.85, upper=1.15)

factor_final = factor_tail.copy()
factor_final.loc[profit_rank <= 1000, :] = factor_top.loc[profit_rank <= 1000, :]

# 4. Apply factor to future dates
pred_weekday = pred_base.copy()

for col in pred_weekday.columns:
    dow = pd.Timestamp(col).dayofweek
    
    if dow == 6:
        pred_weekday[col] = 0.0
    else:
        pred_weekday[col] = pred_weekday[col] * factor_final[dow]

# 5. Existing postprocess
pred_weekday = postprocess_prediction(
    pred_weekday,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_weekday_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_weekday = make_kaggle_submission(
    pred_weekday,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_weekday_path
)

# QA
f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_weekday["id"].str.endswith("_validation")
evaluation_rows = submission_weekday["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_weekday_path)
print("Shape:", submission_weekday.shape)
print("Unique IDs:", submission_weekday["id"].nunique())
print("Missing:", submission_weekday[f_cols].isna().sum().sum())
print("Negative:", (submission_weekday[f_cols] < 0).sum().sum())
print("Total:", submission_weekday[f_cols].sum().sum())
print("Max:", submission_weekday[f_cols].max().max())
print("Validation Sunday total:", submission_weekday.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_weekday.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_weekday.shape == sample.shape
assert submission_weekday["id"].tolist() == sample["id"].tolist()
assert submission_weekday[f_cols].isna().sum().sum() == 0
assert (submission_weekday[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_recent18_21_w50_50_weekday_shrink_v1.csv
Saved: ../outputs/submissions/submission_recent18_21_w50_50_weekday_shrink_v1.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 57187.59722157995
Max: 118.61984126984127
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [43]:
WINDOWS_TO_TEST = [7, 10, 14, 18, 21, 28, 35, 56]

def make_raw_recent_pred_final(window):
    return make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=(window,),
        weights=(1.0,)
    )

raw_final_preds = {}

for w in WINDOWS_TO_TEST:
    raw_final_preds[w] = make_raw_recent_pred_final(w)
    print(f"window {w} total:", raw_final_preds[w].values.sum())

window 7 total: 56360.0
window 10 total: 70554.4
window 14 total: 65408.0
window 18 total: 70563.11111111111
window 21 total: 71282.66666666666
window 28 total: 72936.0
window 35 total: 69801.6
window 56 total: 74413.0


In [44]:
RECENT_CANDIDATES = {
    # Current strong family
    "w18_21_50_50": {
        "windows": (18, 21),
        "weights": (0.50, 0.50)
    },
    "w18_21_28_45_50_05": {
        "windows": (18, 21, 28),
        "weights": (0.45, 0.50, 0.05)
    },
    "w18_21_28_35_50_15": {
        "windows": (18, 21, 28),
        "weights": (0.35, 0.50, 0.15)
    },

    # More aggressive recent
    "w14_18_21_20_45_35": {
        "windows": (14, 18, 21),
        "weights": (0.20, 0.45, 0.35)
    },
    "w14_18_21_30_45_25": {
        "windows": (14, 18, 21),
        "weights": (0.30, 0.45, 0.25)
    },
    "w10_18_21_15_50_35": {
        "windows": (10, 18, 21),
        "weights": (0.15, 0.50, 0.35)
    },

    # Conservative/private-safe
    "w18_21_28_56_30_45_20_05": {
        "windows": (18, 21, 28, 56),
        "weights": (0.30, 0.45, 0.20, 0.05)
    },
    "w21_28_56_35_45_20": {
        "windows": (21, 28, 56),
        "weights": (0.35, 0.45, 0.20)
    },

    # Very short, only for top buckets if CV supports
    "w10_14_18_20_35_45": {
        "windows": (10, 14, 18),
        "weights": (0.20, 0.35, 0.45)
    }
}

In [45]:
def combine_final_candidate(candidate_name):
    cfg = RECENT_CANDIDATES[candidate_name]
    windows = cfg["windows"]
    weights = cfg["weights"]

    pred = None

    for w, weight in zip(windows, weights):
        if pred is None:
            pred = weight * raw_final_preds[w].copy()
        else:
            pred += weight * raw_final_preds[w]

    return pred

In [46]:
folds_for_search = [
    {
        "fold": "recent_2025",
        "train_end": "2025-07-11",
        "valid_start": "2025-07-12",
        "horizon": 56
    },
    {
        "fold": "seasonal_2024",
        "train_end": "2024-09-05",
        "valid_start": "2024-09-06",
        "horizon": 56
    }
]

def make_raw_recent_pred_fold(train_end, valid_start, horizon, window):
    return make_recent_blend_prediction(
        daily_panel,
        train_end=train_end,
        forecast_start=valid_start,
        horizon=horizon,
        target_col=TARGET_FOR_PRED,
        windows=(window,),
        weights=(1.0,)
    )

def combine_fold_candidate(raw_fold_preds, candidate_name):
    cfg = RECENT_CANDIDATES[candidate_name]
    windows = cfg["windows"]
    weights = cfg["weights"]

    pred = None
    for w, weight in zip(windows, weights):
        if pred is None:
            pred = weight * raw_fold_preds[w].copy()
        else:
            pred += weight * raw_fold_preds[w]

    return pred

In [47]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "rank_001_100"
    elif rank <= 500:
        return "rank_101_500"
    elif rank <= 1000:
        return "rank_501_1000"
    elif rank <= 2000:
        return "rank_1001_2000"
    else:
        return "rank_2001_plus"


bucket_records = []

for fold_cfg in folds_for_search:
    fold_name = fold_cfg["fold"]
    train_end = fold_cfg["train_end"]
    valid_start = fold_cfg["valid_start"]
    horizon = fold_cfg["horizon"]

    print("=" * 80)
    print("Fold:", fold_name)

    metric_info_fold = compute_sku_metric_info(
        daily_panel,
        train_end=train_end,
        target_col="y_net"
    )

    metric_info_fold["rank_bucket"] = metric_info_fold["profit_rank"].apply(assign_rank_bucket)

    actual_wide = make_actual_matrix(
        daily_panel,
        start_date=valid_start,
        horizon=horizon,
        target_col="y_net"
    )

    raw_fold_preds = {}
    for w in WINDOWS_TO_TEST:
        raw_fold_preds[w] = make_raw_recent_pred_fold(
            train_end=train_end,
            valid_start=valid_start,
            horizon=horizon,
            window=w
        )

    for cand_name in RECENT_CANDIDATES.keys():
        pred = combine_fold_candidate(raw_fold_preds, cand_name)

        pred = postprocess_prediction(
            pred,
            panel=daily_panel,
            sku_activity=sku_activity,
            train_end=train_end,
            sunday_factor=0.0,
            apply_cap=True
        )

        score, detail = wrmsse_score(
            actual_wide,
            pred,
            metric_info_fold
        )

        detail = detail.merge(
            metric_info_fold[["ItemCode", "rank_bucket"]],
            on="ItemCode",
            how="left"
        )

        bucket_summary = (
            detail.groupby("rank_bucket", as_index=False)
            .agg(
                wrmsse_contribution=("weighted_rmsse", "sum"),
                weight_sum=("weight", "sum"),
                avg_rmsse=("rmsse", "mean")
            )
        )

        for _, row in bucket_summary.iterrows():
            bucket_records.append({
                "fold": fold_name,
                "candidate": cand_name,
                "rank_bucket": row["rank_bucket"],
                "wrmsse_contribution": row["wrmsse_contribution"],
                "weight_sum": row["weight_sum"],
                "avg_rmsse": row["avg_rmsse"],
                "full_score": score
            })

bucket_result = pd.DataFrame(bucket_records)
display(bucket_result.head())

Fold: recent_2025
Fold: seasonal_2024


,fold,candidate,rank_bucket,wrmsse_contribution,weight_sum,avg_rmsse,full_score
0,recent_2025,w18_21_50_50,rank_001_100,0.273992,0.397870,0.467988,0.570869
1,recent_2025,w18_21_50_50,rank_1001_2000,0.047673,0.105264,0.446263,0.570869
2,recent_2025,w18_21_50_50,rank_101_500,0.131748,0.235905,0.562716,0.570869
3,recent_2025,w18_21_50_50,rank_2001_plus,0.058117,0.146865,89.392922,0.570869
4,recent_2025,w18_21_50_50,rank_501_1000,0.059338,0.114096,0.505104,0.570869


In [48]:
bucket_avg = (
    bucket_result
    .groupby(["rank_bucket", "candidate"], as_index=False)
    .agg(
        mean_contribution=("wrmsse_contribution", "mean"),
        mean_full_score=("full_score", "mean")
    )
)

bucket_best_top5 = (
    bucket_avg
    .sort_values(["rank_bucket", "mean_contribution"])
    .groupby("rank_bucket")
    .head(5)
)

display(bucket_best_top5)

,rank_bucket,candidate,mean_contribution,mean_full_score
8,rank_001_100,w21_28_56_35_45_20,0.296750,0.657596
6,rank_001_100,w18_21_28_56_30_45_20_05,0.297858,0.659470
1,rank_001_100,w10_18_21_15_50_35,0.298295,0.660953
4,rank_001_100,w18_21_28_35_50_15,0.298306,0.660239
5,rank_001_100,w18_21_28_45_50_05,0.298523,0.660816
17,rank_1001_2000,w21_28_56_35_45_20,0.055620,0.657596
15,rank_1001_2000,w18_21_28_56_30_45_20_05,0.055728,0.659470
13,rank_1001_2000,w18_21_28_35_50_15,0.055771,0.660239
14,rank_1001_2000,w18_21_28_45_50_05,0.055832,0.660816
16,rank_1001_2000,w18_21_50_50,0.055865,0.661137


In [49]:
pivot_bucket = bucket_avg.pivot_table(
    index="rank_bucket",
    columns="candidate",
    values="mean_contribution"
)

display(pivot_bucket)

best_by_bucket = (
    bucket_avg
    .sort_values(["rank_bucket", "mean_contribution"])
    .groupby("rank_bucket")
    .first()
    .reset_index()
)

display(best_by_bucket)

candidate,w10_14_18_20_35_45,w10_18_21_15_50_35,w14_18_21_20_45_35,w14_18_21_30_45_25,w18_21_28_35_50_15,w18_21_28_45_50_05,w18_21_28_56_30_45_20_05,w18_21_50_50,w21_28_56_35_45_20
rank_bucket,,,,,,,,,
rank_001_100,0.298706,0.298295,0.298691,0.298704,0.298306,0.298523,0.297858,0.298640,0.296750
rank_1001_2000,0.056153,0.055925,0.055926,0.055979,0.055771,0.055832,0.055728,0.055865,0.055620
rank_101_500,0.168572,0.168270,0.168280,0.168342,0.168042,0.168181,0.167930,0.168261,0.167738
rank_2001_plus,0.064612,0.064514,0.064455,0.064474,0.064337,0.064427,0.064256,0.064474,0.063981
rank_501_1000,0.074101,0.073949,0.073903,0.073930,0.073782,0.073854,0.073699,0.073895,0.073506


,rank_bucket,candidate,mean_contribution,mean_full_score
0,rank_001_100,w21_28_56_35_45_20,0.296750,0.657596
1,rank_1001_2000,w21_28_56_35_45_20,0.055620,0.657596
2,rank_101_500,w21_28_56_35_45_20,0.167738,0.657596
3,rank_2001_plus,w21_28_56_35_45_20,0.063981,0.657596
4,rank_501_1000,w21_28_56_35_45_20,0.073506,0.657596


In [ ]:
bucket_avg = (
    bucket_result
    .groupby(["rank_bucket", "candidate"], as_index=False)
    .agg(
        mean_contribution=("wrmsse_contribution", "mean"),
        mean_full_score=("full_score", "mean")
    )
)

bucket_best = (
    bucket_avg
    .sort_values(["rank_bucket", "mean_contribution"])
    .groupby("rank_bucket")
    .head(5)
)

display(bucket_best)

best_by_bucket = (
    bucket_avg
    .sort_values(["rank_bucket", "mean_contribution"])
    .groupby("rank_bucket")
    .first()
    .reset_index()
)

display(best_by_bucket)

In [ ]:
# Map bucket -> selected candidate from local CV
bucket_to_candidate = dict(
    zip(best_by_bucket["rank_bucket"], best_by_bucket["candidate"])
)

print(bucket_to_candidate)

# Start with zeros
item_index = sorted(daily_panel["ItemCode"].unique())
forecast_dates = pd.date_range(FINAL_FORECAST_START, periods=HORIZON, freq="D")

pred_bucket_opt = pd.DataFrame(
    0.0,
    index=item_index,
    columns=forecast_dates
)

meta_final = sku_activity.set_index("ItemCode").reindex(item_index)
profit_rank_final = meta_final["profit_rank"].fillna(999999)
bucket_series = profit_rank_final.apply(assign_rank_bucket)

for bucket, cand_name in bucket_to_candidate.items():
    mask = bucket_series == bucket
    sku_list = bucket_series[mask].index

    pred_cand = combine_final_candidate(cand_name)
    pred_bucket_opt.loc[sku_list, :] = pred_cand.loc[sku_list, :]

pred_bucket_opt = postprocess_prediction(
    pred_bucket_opt,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

candidate_name = "bucket_optimized_recent_cv_v1"

submission_bucket_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_bucket = make_kaggle_submission(
    pred_bucket_opt,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_bucket_path
)

f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_bucket["id"].str.endswith("_validation")
evaluation_rows = submission_bucket["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_bucket_path)
print("Shape:", submission_bucket.shape)
print("Unique IDs:", submission_bucket["id"].nunique())
print("Missing:", submission_bucket[f_cols].isna().sum().sum())
print("Negative:", (submission_bucket[f_cols] < 0).sum().sum())
print("Total:", submission_bucket[f_cols].sum().sum())
print("Max:", submission_bucket[f_cols].max().max())
print("Validation Sunday total:", submission_bucket.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_bucket.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_bucket.shape == sample.shape
assert submission_bucket["id"].tolist() == sample["id"].tolist()
assert submission_bucket[f_cols].isna().sum().sum() == 0
assert (submission_bucket[f_cols] < 0).sum().sum() == 0

In [50]:
# Candidate: only adjust top-profit active SKUs, keep all others as current best.
# Base = current best: 0.50 recent18 + 0.50 recent21

def make_raw_recent_pred(window):
    return make_recent_blend_prediction(
        daily_panel,
        train_end=FINAL_TRAIN_END,
        forecast_start=FINAL_FORECAST_START,
        horizon=HORIZON,
        target_col=TARGET_FOR_PRED,
        windows=(window,),
        weights=(1.0,)
    )

# Reuse if already exists, otherwise create
try:
    pred14_raw
except NameError:
    pred14_raw = make_raw_recent_pred(14)

try:
    pred18_raw
except NameError:
    pred18_raw = make_raw_recent_pred(18)

try:
    pred21_raw
except NameError:
    pred21_raw = make_raw_recent_pred(21)

candidate_name = "top500_active_aggressive_v1"

# Base: current best
pred_top500_v1 = (
    0.50 * pred18_raw +
    0.50 * pred21_raw
)

item_index = pred_top500_v1.index
meta = sku_activity.set_index("ItemCode").reindex(item_index)

profit_rank = meta["profit_rank"].fillna(999999)
days_since = meta["days_since_last_sale"].fillna(9999)
active_days = meta["active_days"].fillna(0)

# Group A: top 100, active recently
mask_top100_active = (
    (profit_rank <= 100) &
    (days_since <= 45) &
    (active_days >= 10)
)

# More responsive, but not too extreme
pred_top500_v1.loc[mask_top100_active, :] = (
    0.25 * pred14_raw.loc[mask_top100_active, :] +
    0.50 * pred18_raw.loc[mask_top100_active, :] +
    0.25 * pred21_raw.loc[mask_top100_active, :]
)

# Group B: rank 101-500, active recently
mask_101_500_active = (
    (profit_rank > 100) &
    (profit_rank <= 500) &
    (days_since <= 45) &
    (active_days >= 8)
)

pred_top500_v1.loc[mask_101_500_active, :] = (
    0.15 * pred14_raw.loc[mask_101_500_active, :] +
    0.50 * pred18_raw.loc[mask_101_500_active, :] +
    0.35 * pred21_raw.loc[mask_101_500_active, :]
)

# Existing postprocess
pred_top500_v1 = postprocess_prediction(
    pred_top500_v1,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_top500_v1_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_top500_v1 = make_kaggle_submission(
    pred_top500_v1,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_top500_v1_path
)

# QA
f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_top500_v1["id"].str.endswith("_validation")
evaluation_rows = submission_top500_v1["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_top500_v1_path)
print("Shape:", submission_top500_v1.shape)
print("Unique IDs:", submission_top500_v1["id"].nunique())
print("Missing:", submission_top500_v1[f_cols].isna().sum().sum())
print("Negative:", (submission_top500_v1[f_cols] < 0).sum().sum())
print("Total:", submission_top500_v1[f_cols].sum().sum())
print("Max:", submission_top500_v1[f_cols].max().max())
print("Validation Sunday total:", submission_top500_v1.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_top500_v1.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_top500_v1.shape == sample.shape
assert submission_top500_v1["id"].tolist() == sample["id"].tolist()
assert submission_top500_v1[f_cols].isna().sum().sum() == 0
assert (submission_top500_v1[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_top500_active_aggressive_v1.csv
Saved: ../outputs/submissions/submission_top500_active_aggressive_v1.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 60244.99047619046
Max: 89.17460317460318
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [51]:
# Compare candidate vs current best prediction matrix
base_pred = pred_base  # hoặc pred_w50_50 nếu bạn đặt tên khác
cand_pred = pred_top500_v1

delta = cand_pred - base_pred
abs_delta_by_sku = delta.abs().sum(axis=1)

changed_skus = abs_delta_by_sku[abs_delta_by_sku > 1e-9].index

meta_check = sku_activity.set_index("ItemCode").reindex(cand_pred.index)

print("Changed SKUs:", len(changed_skus))
print("Total base:", base_pred.values.sum())
print("Total candidate:", cand_pred.values.sum())
print("Total delta:", cand_pred.values.sum() - base_pred.values.sum())
print("Max abs delta:", delta.abs().max().max())

display(
    meta_check.loc[changed_skus]
    .assign(abs_delta=abs_delta_by_sku.loc[changed_skus])
    .sort_values("profit_rank")
    [["profit_rank", "active_days", "days_since_last_sale", "abs_delta"]]
    .head(30)
)

Changed SKUs: 2358
Total base: 70922.88888888889
Total candidate: 60244.990476190476
Total delta: -10677.898412698414
Max abs delta: 91.24603174603175


,profit_rank,active_days,days_since_last_sale,abs_delta
SKU-00003,1,1061,1,113.746032
SKU-00002,2,895,0,25.142857
SKU-12534,6,657,1,44.317460
SKU-09760,7,994,0,829.396825
SKU-14323,10,846,0,205.333333
SKU-14320,11,897,0,148.698413
SKU-06780,13,785,14,1.968254
SKU-06877,15,766,14,2.952381
SKU-12176,16,970,0,123.396825
SKU-08863,17,707,8,8.380952


In [52]:
candidate_name = "recent14_18_21_w20_45_35_sunday0"

pred_w14_18_21 = make_recent_blend_prediction(
    daily_panel,
    train_end=FINAL_TRAIN_END,
    forecast_start=FINAL_FORECAST_START,
    horizon=HORIZON,
    target_col=TARGET_FOR_PRED,
    windows=(14, 18, 21),
    weights=(0.20, 0.45, 0.35)
)

pred_w14_18_21 = postprocess_prediction(
    pred_w14_18_21,
    panel=daily_panel,
    sku_activity=sku_activity,
    train_end=FINAL_TRAIN_END,
    sunday_factor=0.0,
    apply_cap=True
)

submission_w14_18_21_path = SUB_DIR / f"submission_{candidate_name}.csv"

submission_w14_18_21 = make_kaggle_submission(
    pred_w14_18_21,
    sample=sample,
    forecast_start=FINAL_FORECAST_START,
    output_path=submission_w14_18_21_path
)

f_cols = [f"F{i}" for i in range(1, 29)]
validation_rows = submission_w14_18_21["id"].str.endswith("_validation")
evaluation_rows = submission_w14_18_21["id"].str.endswith("_evaluation")
sunday_f = ["F2", "F9", "F16", "F23"]

print("Saved:", submission_w14_18_21_path)
print("Shape:", submission_w14_18_21.shape)
print("Unique IDs:", submission_w14_18_21["id"].nunique())
print("Missing:", submission_w14_18_21[f_cols].isna().sum().sum())
print("Negative:", (submission_w14_18_21[f_cols] < 0).sum().sum())
print("Total:", submission_w14_18_21[f_cols].sum().sum())
print("Max:", submission_w14_18_21[f_cols].max().max())
print("Validation Sunday total:", submission_w14_18_21.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", submission_w14_18_21.loc[evaluation_rows, sunday_f].sum().sum())

assert submission_w14_18_21.shape == sample.shape
assert submission_w14_18_21["id"].tolist() == sample["id"].tolist()
assert submission_w14_18_21[f_cols].isna().sum().sum() == 0
assert (submission_w14_18_21[f_cols] < 0).sum().sum() == 0

Saved: ../outputs/submissions/submission_recent14_18_21_w20_45_35_sunday0.csv
Saved: ../outputs/submissions/submission_recent14_18_21_w20_45_35_sunday0.csv
Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 59814.79999999999
Max: 89.83571428571427
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0


In [53]:
candidate_specs_for_audit = {
    "w35_50_15": {
        "windows": (18, 21, 28),
        "weights": (0.35, 0.50, 0.15)
    },
    "w45_50_05": {
        "windows": (18, 21, 28),
        "weights": (0.45, 0.50, 0.05)
    },
    "w50_50": {
        "windows": (18, 21),
        "weights": (0.50, 0.50)
    },
    "recent14_18_21_w20_45_35": {
        "windows": (14, 18, 21),
        "weights": (0.20, 0.45, 0.35)
    },
    "recent14_18_21_w30_45_25": {
        "windows": (14, 18, 21),
        "weights": (0.30, 0.45, 0.25)
    },
    "private_safe_18_21_28_56": {
        "windows": (18, 21, 28, 56),
        "weights": (0.30, 0.45, 0.20, 0.05)
    },
}

In [54]:
audit_fold = {
    "fold": "recent_2025",
    "train_end": "2025-07-11",
    "valid_start": "2025-07-12",
    "horizon": 56
}

metric_info_audit = compute_sku_metric_info(
    daily_panel,
    train_end=audit_fold["train_end"],
    target_col="y_net"
)

actual_audit = make_actual_matrix(
    daily_panel,
    start_date=audit_fold["valid_start"],
    horizon=audit_fold["horizon"],
    target_col="y_net"
)

def make_candidate_pred_for_fold(spec, fold_cfg):
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=fold_cfg["train_end"],
        forecast_start=fold_cfg["valid_start"],
        horizon=fold_cfg["horizon"],
        target_col=TARGET_FOR_PRED,
        windows=spec["windows"],
        weights=spec["weights"]
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=fold_cfg["train_end"],
        sunday_factor=0.0,
        apply_cap=True
    )

    return pred

audit_records = []
audit_details = {}
audit_preds = {}

for name, spec in candidate_specs_for_audit.items():
    pred = make_candidate_pred_for_fold(spec, audit_fold)
    score, detail = wrmsse_score(actual_audit, pred, metric_info_audit)

    audit_records.append({
        "candidate": name,
        "wrmsse_recent_2025": score,
        "windows": str(spec["windows"]),
        "weights": str(spec["weights"]),
        "total_pred": pred.values.sum(),
        "max_pred": pred.max().max()
    })

    audit_details[name] = detail
    audit_preds[name] = pred

audit_score_df = pd.DataFrame(audit_records).sort_values("wrmsse_recent_2025")
display(audit_score_df)

,candidate,wrmsse_recent_2025,windows,weights,total_pred,max_pred
5,private_safe_18_21_28_56,0.569484,"(18, 21, 28, 56)","(0.3, 0.45, 0.2, 0.05)",65239.614286,107.493155
0,w35_50_15,0.569991,"(18, 21, 28)","(0.35, 0.5, 0.15)",65290.361905,104.624008
1,w45_50_05,0.570560,"(18, 21, 28)","(0.45, 0.5, 0.05)",65786.057143,101.729167
3,recent14_18_21_w20_45_35,0.570816,"(14, 18, 21)","(0.2, 0.45, 0.35)",66952.800000,101.920238
2,w50_50,0.570869,"(18, 21)","(0.5, 0.5)",66033.904762,100.281746
4,recent14_18_21_w30_45_25,0.570902,"(14, 18, 21)","(0.3, 0.45, 0.25)",67587.085714,103.204762


In [55]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "rank_001_100"
    elif rank <= 500:
        return "rank_101_500"
    elif rank <= 1000:
        return "rank_501_1000"
    elif rank <= 2000:
        return "rank_1001_2000"
    else:
        return "rank_2001_plus"

bucket_compare_records = []

for name, detail in audit_details.items():
    d = detail.copy()
    d["rank_bucket"] = d["profit_rank"].apply(assign_rank_bucket)

    bucket_summary = (
        d.groupby("rank_bucket", as_index=False)
        .agg(
            wrmsse_contribution=("weighted_rmsse", "sum"),
            weight_sum=("weight", "sum"),
            avg_rmsse=("rmsse", "mean")
        )
    )

    bucket_summary["candidate"] = name
    bucket_compare_records.append(bucket_summary)

bucket_compare_df = pd.concat(bucket_compare_records, ignore_index=True)

display(
    bucket_compare_df
    .pivot_table(
        index="rank_bucket",
        columns="candidate",
        values="wrmsse_contribution"
    )
)

candidate,private_safe_18_21_28_56,recent14_18_21_w20_45_35,recent14_18_21_w30_45_25,w35_50_15,w45_50_05,w50_50
rank_bucket,,,,,,
rank_001_100,0.273575,0.273611,0.273409,0.273765,0.273911,0.273992
rank_1001_2000,0.047487,0.047777,0.047851,0.047548,0.047630,0.047673
rank_101_500,0.131412,0.131867,0.131976,0.131515,0.131666,0.131748
rank_2001_plus,0.057862,0.058134,0.058171,0.057949,0.058059,0.058117
rank_501_1000,0.059148,0.059426,0.059496,0.059214,0.059294,0.059338


In [56]:
def horizon_error_summary(actual_wide, pred_wide, metric_info):
    actual = actual_wide.sort_index()
    pred = pred_wide.reindex(index=actual.index, columns=actual.columns).fillna(0)

    metric = metric_info.set_index("ItemCode").reindex(actual.index)
    weights = metric["weight"].to_numpy()

    rows = []

    for i, col in enumerate(actual.columns):
        err_sq = (actual[col].to_numpy() - pred[col].to_numpy()) ** 2
        weighted_mse_proxy = np.sum(weights * err_sq)

        rows.append({
            "horizon": i + 1,
            "date": col,
            "dayofweek": pd.Timestamp(col).dayofweek,
            "weighted_mse_proxy": weighted_mse_proxy,
            "actual_total": actual[col].sum(),
            "pred_total": pred[col].sum(),
            "delta_total": pred[col].sum() - actual[col].sum()
        })

    return pd.DataFrame(rows)

horizon_summaries = []

for name, pred in audit_preds.items():
    hsum = horizon_error_summary(actual_audit, pred, metric_info_audit)
    hsum["candidate"] = name
    horizon_summaries.append(hsum)

horizon_compare = pd.concat(horizon_summaries, ignore_index=True)

display(
    horizon_compare
    .pivot_table(
        index="horizon",
        columns="candidate",
        values="delta_total"
    )
    .head(56)
)

candidate,private_safe_18_21_28_56,recent14_18_21_w20_45_35,recent14_18_21_w30_45_25,w35_50_15,w45_50_05,w50_50
horizon,,,,,,
1,-211.841369,-176.15,-162.935714,-210.784127,-200.457143,-195.293651
2,0.000000,0.00,0.000000,0.000000,0.000000,0.000000
3,-610.841369,-575.15,-561.935714,-609.784127,-599.457143,-594.293651
4,232.158631,267.85,281.064286,233.215873,243.542857,248.706349
5,-1945.841369,-1910.15,-1896.935714,-1944.784127,-1934.457143,-1929.293651
6,174.158631,209.85,223.064286,175.215873,185.542857,190.706349
7,-255.841369,-220.15,-206.935714,-254.784127,-244.457143,-239.293651
8,684.158631,719.85,733.064286,685.215873,695.542857,700.706349
9,0.000000,0.00,0.000000,0.000000,0.000000,0.000000


In [57]:
best_name = "w50_50"

h = horizon_error_summary(actual_audit, audit_preds[best_name], metric_info_audit)

print("Actual total:", h["actual_total"].sum())
print("Pred total:", h["pred_total"].sum())
print("Pred / Actual:", h["pred_total"].sum() / h["actual_total"].sum())

display(h[["horizon", "date", "dayofweek", "actual_total", "pred_total", "delta_total"]])

Actual total: 73131.0
Pred total: 66033.90476190478
Pred / Actual: 0.9029536689215897


,horizon,date,dayofweek,actual_total,pred_total,delta_total
0,1,2025-07-12,5,1571.0,1375.706349,-195.293651
1,2,2025-07-13,6,0.0,0.000000,0.000000
2,3,2025-07-14,0,1970.0,1375.706349,-594.293651
3,4,2025-07-15,1,1127.0,1375.706349,248.706349
4,5,2025-07-16,2,3305.0,1375.706349,-1929.293651
5,6,2025-07-17,3,1185.0,1375.706349,190.706349
6,7,2025-07-18,4,1615.0,1375.706349,-239.293651
7,8,2025-07-19,5,675.0,1375.706349,700.706349
8,9,2025-07-20,6,0.0,0.000000,0.000000
9,10,2025-07-21,0,1760.0,1375.706349,-384.293651


In [58]:
# =========================
# PATTERN AUDIT SETUP
# =========================

AUDIT_FOLD = {
    "fold": "recent_2025",
    "train_end": "2025-07-11",
    "valid_start": "2025-07-12",
    "horizon": 56
}

metric_info_audit = compute_sku_metric_info(
    daily_panel,
    train_end=AUDIT_FOLD["train_end"],
    target_col="y_net"
)

actual_audit = make_actual_matrix(
    daily_panel,
    start_date=AUDIT_FOLD["valid_start"],
    horizon=AUDIT_FOLD["horizon"],
    target_col="y_net"
)

print("actual_audit:", actual_audit.shape)
print("metric_info:", metric_info_audit.shape)

actual_audit: (15972, 56)
metric_info: (15972, 9)


In [59]:
# =========================
# CANDIDATES FOR AUDIT ONLY
# =========================

AUDIT_CANDIDATES = {
    # Current best family
    "w50_50": {
        "windows": (18, 21),
        "weights": (0.50, 0.50)
    },
    "w45_50_05": {
        "windows": (18, 21, 28),
        "weights": (0.45, 0.50, 0.05)
    },
    "w35_50_15": {
        "windows": (18, 21, 28),
        "weights": (0.35, 0.50, 0.15)
    },

    # More recent
    "w14_18_21_20_45_35": {
        "windows": (14, 18, 21),
        "weights": (0.20, 0.45, 0.35)
    },
    "w14_18_21_30_45_25": {
        "windows": (14, 18, 21),
        "weights": (0.30, 0.45, 0.25)
    },
    "w10_14_18_20_35_45": {
        "windows": (10, 14, 18),
        "weights": (0.20, 0.35, 0.45)
    },

    # More conservative
    "w18_21_28_56_private": {
        "windows": (18, 21, 28, 56),
        "weights": (0.30, 0.45, 0.20, 0.05)
    },
    "w21_28_56_private": {
        "windows": (21, 28, 56),
        "weights": (0.35, 0.45, 0.20)
    },
}

In [60]:
def make_candidate_pred_for_fold(spec, fold_cfg):
    pred = make_recent_blend_prediction(
        daily_panel,
        train_end=fold_cfg["train_end"],
        forecast_start=fold_cfg["valid_start"],
        horizon=fold_cfg["horizon"],
        target_col=TARGET_FOR_PRED,
        windows=spec["windows"],
        weights=spec["weights"]
    )

    pred = postprocess_prediction(
        pred,
        panel=daily_panel,
        sku_activity=sku_activity,
        train_end=fold_cfg["train_end"],
        sunday_factor=0.0,
        apply_cap=True
    )

    return pred


audit_records = []
audit_details = {}
audit_preds = {}

for name, spec in AUDIT_CANDIDATES.items():
    print("Evaluating:", name)

    pred = make_candidate_pred_for_fold(spec, AUDIT_FOLD)
    score, detail = wrmsse_score(actual_audit, pred, metric_info_audit)

    audit_records.append({
        "candidate": name,
        "local_wrmsse": score,
        "windows": str(spec["windows"]),
        "weights": str(spec["weights"]),
        "total_pred": pred.values.sum(),
        "max_pred": pred.max().max()
    })

    audit_details[name] = detail
    audit_preds[name] = pred

audit_score_df = pd.DataFrame(audit_records).sort_values("local_wrmsse")
display(audit_score_df)

Evaluating: w50_50
Evaluating: w45_50_05
Evaluating: w35_50_15
Evaluating: w14_18_21_20_45_35
Evaluating: w14_18_21_30_45_25
Evaluating: w10_14_18_20_35_45
Evaluating: w18_21_28_56_private
Evaluating: w21_28_56_private


,candidate,local_wrmsse,windows,weights,total_pred,max_pred
7,w21_28_56_private,0.568196,"(21, 28, 56)","(0.35, 0.45, 0.2)",64242.000000,118.429167
6,w18_21_28_56_private,0.569484,"(18, 21, 28, 56)","(0.3, 0.45, 0.2, 0.05)",65239.614286,107.493155
2,w35_50_15,0.569991,"(18, 21, 28)","(0.35, 0.5, 0.15)",65290.361905,104.624008
1,w45_50_05,0.570560,"(18, 21, 28)","(0.45, 0.5, 0.05)",65786.057143,101.729167
3,w14_18_21_20_45_35,0.570816,"(14, 18, 21)","(0.2, 0.45, 0.35)",66952.800000,101.920238
0,w50_50,0.570869,"(18, 21)","(0.5, 0.5)",66033.904762,100.281746
4,w14_18_21_30_45_25,0.570902,"(14, 18, 21)","(0.3, 0.45, 0.25)",67587.085714,103.204762
5,w10_14_18_20_35_45,0.571462,"(10, 14, 18)","(0.2, 0.35, 0.45)",69298.560000,109.390000


In [61]:
def assign_rank_bucket(rank):
    if rank <= 100:
        return "rank_001_100"
    elif rank <= 500:
        return "rank_101_500"
    elif rank <= 1000:
        return "rank_501_1000"
    elif rank <= 2000:
        return "rank_1001_2000"
    else:
        return "rank_2001_plus"


bucket_records = []

for name, detail in audit_details.items():
    d = detail.copy()
    d["rank_bucket"] = d["profit_rank"].apply(assign_rank_bucket)

    bucket_summary = (
        d.groupby("rank_bucket", as_index=False)
        .agg(
            wrmsse_contribution=("weighted_rmsse", "sum"),
            weight_sum=("weight", "sum"),
            avg_rmsse=("rmsse", "mean")
        )
    )

    bucket_summary["candidate"] = name
    bucket_records.append(bucket_summary)

bucket_compare_df = pd.concat(bucket_records, ignore_index=True)

bucket_pivot = bucket_compare_df.pivot_table(
    index="rank_bucket",
    columns="candidate",
    values="wrmsse_contribution"
)

display(bucket_pivot)

candidate,w10_14_18_20_35_45,w14_18_21_20_45_35,w14_18_21_30_45_25,w18_21_28_56_private,w21_28_56_private,w35_50_15,w45_50_05,w50_50
rank_bucket,,,,,,,,
rank_001_100,0.272946,0.273611,0.273409,0.273575,0.273237,0.273765,0.273911,0.273992
rank_1001_2000,0.048096,0.047777,0.047851,0.047487,0.047299,0.047548,0.047630,0.047673
rank_101_500,0.132264,0.131867,0.131976,0.131412,0.131160,0.131515,0.131666,0.131748
rank_2001_plus,0.058413,0.058134,0.058171,0.057862,0.057537,0.057949,0.058059,0.058117
rank_501_1000,0.059742,0.059426,0.059496,0.059148,0.058965,0.059214,0.059294,0.059338


In [62]:
BASE_NAME = "w50_50"

base_detail = audit_details[BASE_NAME].copy()
base_pred = audit_preds[BASE_NAME].copy()

# Add actual/pred totals by SKU
base_detail["actual_total_56"] = actual_audit.sum(axis=1).reindex(base_detail["ItemCode"]).to_numpy()
base_detail["pred_total_56"] = base_pred.sum(axis=1).reindex(base_detail["ItemCode"]).to_numpy()
base_detail["pred_minus_actual"] = base_detail["pred_total_56"] - base_detail["actual_total_56"]
base_detail["pred_actual_ratio"] = np.where(
    base_detail["actual_total_56"] > 0,
    base_detail["pred_total_56"] / base_detail["actual_total_56"],
    np.nan
)

top_error_skus = (
    base_detail
    .sort_values("weighted_rmsse", ascending=False)
    .head(100)
)

display(
    top_error_skus[
        [
            "ItemCode",
            "profit_rank",
            "weight",
            "actual_total_56",
            "pred_total_56",
            "pred_minus_actual",
            "pred_actual_ratio",
            "rmsse",
            "weighted_rmsse"
        ]
    ]
)

,ItemCode,profit_rank,weight,actual_total_56,pred_total_56,pred_minus_actual,pred_actual_ratio,rmsse,weighted_rmsse
2,SKU-00003,1,0.095070,628.0,482.095238,-145.904762,0.767668,1.287822,0.122433
1,SKU-00002,2,0.045248,324.0,416.190476,92.190476,1.284539,1.178185,0.053310
13993,SKU-14323,10,0.005414,1327.0,657.904762,-669.095238,0.495784,1.872712,0.010139
15241,SKU-15599,26,0.002707,1467.0,292.190476,-1174.809524,0.199176,2.381678,0.006447
13990,SKU-14320,11,0.005120,953.0,396.000000,-557.000000,0.415530,1.156674,0.005922
...,...,...,...,...,...,...,...,...,...
2808,SKU-02892,249,0.000614,34.0,4.952381,-29.047619,0.145658,1.127681,0.000692
1427,SKU-01470,156,0.000881,38.0,42.095238,4.095238,1.107769,0.779797,0.000687
13957,SKU-14286,178,0.000796,93.0,148.571429,55.571429,1.597542,0.861731,0.000686
3581,SKU-03687,110,0.001144,560.0,43.428571,-516.571429,0.077551,0.594668,0.000680


In [63]:
print("Top 100 error actual total:", top_error_skus["actual_total_56"].sum())
print("Top 100 error pred total:", top_error_skus["pred_total_56"].sum())
print("Top 100 pred/actual:", top_error_skus["pred_total_56"].sum() / top_error_skus["actual_total_56"].sum())

print("\nCount underforecast:", (top_error_skus["pred_minus_actual"] < 0).sum())
print("Count overforecast:", (top_error_skus["pred_minus_actual"] > 0).sum())

Top 100 error actual total: 31870.0
Top 100 error pred total: 24798.85714285714
Top 100 pred/actual: 0.7781254202339862

Count underforecast: 69
Count overforecast: 31


In [64]:
top_error_items = top_error_skus["ItemCode"].tolist()

sku_candidate_records = []

for name, detail in audit_details.items():
    d = detail[detail["ItemCode"].isin(top_error_items)].copy()
    d["candidate"] = name
    sku_candidate_records.append(
        d[["ItemCode", "candidate", "rmsse", "weighted_rmsse", "profit_rank", "weight"]]
    )

sku_candidate_df = pd.concat(sku_candidate_records, ignore_index=True)

best_candidate_per_top_error_sku = (
    sku_candidate_df
    .sort_values(["ItemCode", "weighted_rmsse"])
    .groupby("ItemCode")
    .first()
    .reset_index()
)

display(
    best_candidate_per_top_error_sku
    .groupby("candidate", as_index=False)
    .agg(
        sku_count=("ItemCode", "count"),
        total_weight=("weight", "sum"),
        median_profit_rank=("profit_rank", "median")
    )
    .sort_values("total_weight", ascending=False)
)

,candidate,sku_count,total_weight,median_profit_rank
3,w18_21_28_56_private,4,0.098539,91.5
0,w10_14_18_20_35_45,31,0.097287,74.0
4,w21_28_56_private,44,0.059862,103.5
5,w35_50_15,5,0.018528,31.0
6,w50_50,13,0.016836,246.0
2,w14_18_21_30_45_25,2,0.003252,126.0
1,w14_18_21_20_45_35,1,0.001868,54.0


In [65]:
def horizon_error_summary(actual_wide, pred_wide, metric_info):
    actual = actual_wide.sort_index()
    pred = pred_wide.reindex(index=actual.index, columns=actual.columns).fillna(0)

    metric = metric_info.set_index("ItemCode").reindex(actual.index)
    weights = metric["weight"].to_numpy()

    rows = []

    for i, col in enumerate(actual.columns):
        actual_values = actual[col].to_numpy()
        pred_values = pred[col].to_numpy()

        err_sq = (actual_values - pred_values) ** 2
        weighted_mse_proxy = np.sum(weights * err_sq)

        rows.append({
            "horizon": i + 1,
            "date": col,
            "dayofweek": pd.Timestamp(col).dayofweek,
            "actual_total": actual[col].sum(),
            "pred_total": pred[col].sum(),
            "delta_total": pred[col].sum() - actual[col].sum(),
            "weighted_mse_proxy": weighted_mse_proxy
        })

    return pd.DataFrame(rows)


horizon_records = []

for name, pred in audit_preds.items():
    h = horizon_error_summary(actual_audit, pred, metric_info_audit)
    h["candidate"] = name
    horizon_records.append(h)

horizon_compare = pd.concat(horizon_records, ignore_index=True)

# Summary first 28 vs second 28
horizon_compare["half"] = np.where(horizon_compare["horizon"] <= 28, "first_28", "second_28")

half_summary = (
    horizon_compare
    .groupby(["candidate", "half"], as_index=False)
    .agg(
        actual_total=("actual_total", "sum"),
        pred_total=("pred_total", "sum"),
        weighted_mse_proxy=("weighted_mse_proxy", "sum")
    )
)

half_summary["pred_actual_ratio"] = half_summary["pred_total"] / half_summary["actual_total"]

display(
    half_summary
    .sort_values(["half", "weighted_mse_proxy"])
)

,candidate,half,actual_total,pred_total,weighted_mse_proxy,pred_actual_ratio
10,w35_50_15,first_28,37182.0,32645.180952,7492.581452,0.877983
12,w45_50_05,first_28,37182.0,32893.028571,7493.086734,0.884649
14,w50_50,first_28,37182.0,33016.952381,7494.949094,0.887982
6,w18_21_28_56_private,first_28,37182.0,32619.807143,7495.298429,0.877301
2,w14_18_21_20_45_35,first_28,37182.0,33476.400000,7520.812848,0.900339
4,w14_18_21_30_45_25,first_28,37182.0,33793.542857,7537.591595,0.908868
8,w21_28_56_private,first_28,37182.0,32121.000000,7541.474150,0.863886
0,w10_14_18_20_35_45,first_28,37182.0,34649.280000,7546.516048,0.931883
7,w18_21_28_56_private,second_28,35949.0,32619.807143,7693.914924,0.907391
11,w35_50_15,second_28,35949.0,32645.180952,7696.170484,0.908097
